In [1]:
# Cell 01 — Configure Architecture-A model, dual-mode training, and early stopping
# DECISION: UPDATE

from pathlib import Path
import math
import hashlib
import json

# =========================================================================
# RUN MODE
# =========================================================================
# Valid values:
#   "fast"       -> short pipeline validation
#   "production" -> real Architecture-A experiment
RUN_MODE = "production"

assert RUN_MODE in {
    "fast",
    "production",
}, (
    f"Unsupported RUN_MODE={RUN_MODE!r}. "
    "Use 'fast' or 'production'."
)

# =========================================================================
# EXPERIMENT IDENTITY
# =========================================================================
ARCHITECTURE_NAME = (
    "Architecture-A-Decoder-Only-Transformer"
)

ARCHITECTURE_VERSION = (
    "A-v2-early-stop-dual-mode"
)

EXPERIMENT_ID = (
    "CS781-P1-A"
    "-decoder-only-transformer"
    "-v2"
)

# =========================================================================
# MODEL CONFIGURATION
# =========================================================================
if RUN_MODE == "fast":

    MODEL_CONFIG = {
        "vocab_size": 50_000,
        "context_length": 128,
        "d_model": 256,
        "n_layers": 2,
        "n_heads": 4,
        "d_ff": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "norm_type": "layernorm",
        "pre_norm": True,
        "position_type": "learned",
        "tie_embeddings": True,
        "use_bias": True,
    }

else:

    MODEL_CONFIG = {
        "vocab_size": 50_000,
        "context_length": 2048,
        "d_model": 768,
        "n_layers": 12,
        "n_heads": 12,
        "d_ff": 3072,
        "dropout": 0.10,
        "activation": "gelu",
        "norm_type": "layernorm",
        "pre_norm": True,
        "position_type": "learned",
        "tie_embeddings": True,
        "use_bias": True,
    }

# =========================================================================
# TRAINING BUDGET
# =========================================================================
if RUN_MODE == "fast":

    TRAIN_TOKENS = 10_240
    VAL_TOKENS = 8_192

    MICRO_BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 2

    EVAL_EVERY_STEPS = 5
    CHECKPOINT_EVERY_STEPS = 5
    LOG_EVERY_STEPS = 1

    # Fast mode only needs enough checks to prove that the
    # early-stopping code path works.
    EARLY_STOPPING_ENABLED = True
    MIN_BEST_CHECKPOINTS_BEFORE_STOP = 2
    MAX_BEST_CHECKPOINTS = 4
    EARLY_STOPPING_PATIENCE = 2
    EARLY_STOPPING_MIN_DELTA = 0.01
    MIN_STEPS_BEFORE_EARLY_STOP = 5

else:

    # ---------------------------------------------------------------------
    # IMPORTANT:
    # This is a ceiling, NOT a target that must be consumed.
    # Early stopping can terminate much earlier.
    # ---------------------------------------------------------------------
    TRAIN_TOKENS = 100_000_000
    VAL_TOKENS = 5_000_000

    MICRO_BATCH_SIZE = 1
    GRAD_ACCUM_STEPS = 8

    EVAL_EVERY_STEPS = 500
    CHECKPOINT_EVERY_STEPS = 500
    LOG_EVERY_STEPS = 10

    # ---------------------------------------------------------------------
    # Early-stopping policy
    # ---------------------------------------------------------------------
    EARLY_STOPPING_ENABLED = True

    # At least eight genuine best-checkpoint improvements are
    # allowed before plateau-based stopping becomes active.
    MIN_BEST_CHECKPOINTS_BEFORE_STOP = 8

    # Absolute protection against a continuously improving curve.
    # The run cannot save more than this many "best" checkpoints.
    MAX_BEST_CHECKPOINTS = 12

    # Stop after this many consecutive validation evaluations without
    # a meaningful improvement.
    EARLY_STOPPING_PATIENCE = 4

    # Improvements smaller than this are treated as negligible.
    # This is deliberately measured in validation-loss units.
    EARLY_STOPPING_MIN_DELTA = 0.005

    # Do not allow early stopping too early in training.
    MIN_STEPS_BEFORE_EARLY_STOP = 2_000

# =========================================================================
# DERIVED TRAINING LIMITS
# =========================================================================
TOKENS_PER_OPTIMIZER_STEP = (
    MICRO_BATCH_SIZE
    * GRAD_ACCUM_STEPS
    * MODEL_CONFIG["context_length"]
)

MAX_STEPS_FROM_TOKEN_BUDGET = max(
    1,
    math.ceil(
        TRAIN_TOKENS
        / TOKENS_PER_OPTIMIZER_STEP
    ),
)

MAX_STEPS = (
    MAX_STEPS_FROM_TOKEN_BUDGET
)

# =========================================================================
# OPTIMIZATION
# =========================================================================
OPTIMIZER_NAME = "AdamW"

OPTIMIZER_CONFIG = {
    "lr": 3e-4,
    "min_lr": 3e-5,
    "betas": (0.9, 0.95),
    "weight_decay": 0.1,
    "grad_clip_norm": 1.0,
}

# Warmup remains proportional to the training budget.
if RUN_MODE == "fast":
    WARMUP_STEPS = 5
else:
    WARMUP_STEPS = min(
        500,
        max(
            100,
            MAX_STEPS // 20,
        ),
    )

# =========================================================================
# CHECKPOINT / RESUME POLICY
# =========================================================================
CHECKPOINT_SCHEMA_VERSION = 2

CHECKPOINT_POLICY = {
    "save_latest": True,
    "save_best": True,
    "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
    "max_best_checkpoints": MAX_BEST_CHECKPOINTS,
    "resume_compatible_only": True,
}

# =========================================================================
# RANDOM SEED
# =========================================================================
SEED = 42

# =========================================================================
# DIRECTORY STRUCTURE
# =========================================================================
NOTEBOOK_DIR = Path(
    "/kaggle/working/architecture_A"
)

CHECKPOINTS_DIR = (
    NOTEBOOK_DIR / "checkpoints"
)

METRICS_DIR = (
    NOTEBOOK_DIR / "metrics"
)

PREDICTIONS_DIR = (
    NOTEBOOK_DIR / "predictions"
)

SUBMISSION_DIR = (
    NOTEBOOK_DIR / "submission"
)

CONFIGS_DIR = (
    NOTEBOOK_DIR / "configs"
)

for _directory in (
    NOTEBOOK_DIR,
    CHECKPOINTS_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    SUBMISSION_DIR,
    CONFIGS_DIR,
):
    _directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# =========================================================================
# CONFIGURATION FINGERPRINT
# =========================================================================
# This is used later to reject checkpoints created under an incompatible
# architecture/training configuration.
CONFIG_FINGERPRINT_PAYLOAD = {
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "experiment_id": EXPERIMENT_ID,
    "run_mode": RUN_MODE,
    "model_config": MODEL_CONFIG,
    "training": {
        "train_tokens": TRAIN_TOKENS,
        "val_tokens": VAL_TOKENS,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "eval_every_steps": EVAL_EVERY_STEPS,
        "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
        "max_steps": MAX_STEPS,
    },
    "optimizer": OPTIMIZER_CONFIG,
    "early_stopping": {
        "enabled": EARLY_STOPPING_ENABLED,
        "min_best_checkpoints_before_stop":
            MIN_BEST_CHECKPOINTS_BEFORE_STOP,
        "max_best_checkpoints":
            MAX_BEST_CHECKPOINTS,
        "patience":
            EARLY_STOPPING_PATIENCE,
        "min_delta":
            EARLY_STOPPING_MIN_DELTA,
        "min_steps":
            MIN_STEPS_BEFORE_EARLY_STOP,
    },
    "seed": SEED,
}

CONFIG_FINGERPRINT = hashlib.sha256(
    json.dumps(
        CONFIG_FINGERPRINT_PAYLOAD,
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()

# =========================================================================
# CONFIG SUMMARY
# =========================================================================
print("✅ Architecture-A configuration loaded.")
print()
print(f"Run mode                 : {RUN_MODE}")
print(f"Architecture             : {ARCHITECTURE_NAME}")
print(f"Context length           : {MODEL_CONFIG['context_length']}")
print(f"d_model                  : {MODEL_CONFIG['d_model']}")
print(f"Layers                   : {MODEL_CONFIG['n_layers']}")
print(f"Heads                    : {MODEL_CONFIG['n_heads']}")
print(f"Feed-forward dim         : {MODEL_CONFIG['d_ff']}")
print(f"Train token ceiling      : {TRAIN_TOKENS:,}")
print(f"Validation tokens        : {VAL_TOKENS:,}")
print(f"Tokens / optimizer step  : {TOKENS_PER_OPTIMIZER_STEP:,}")
print(f"Maximum optimizer steps  : {MAX_STEPS:,}")
print()
print("Early stopping")
print(f"  Enabled                : {EARLY_STOPPING_ENABLED}")
print(
    "  Min best checkpoints   : "
    f"{MIN_BEST_CHECKPOINTS_BEFORE_STOP}"
)
print(
    "  Max best checkpoints   : "
    f"{MAX_BEST_CHECKPOINTS}"
)
print(
    "  Patience               : "
    f"{EARLY_STOPPING_PATIENCE}"
)
print(
    "  Minimum meaningful Δ   : "
    f"{EARLY_STOPPING_MIN_DELTA}"
)
print(
    "  Minimum steps          : "
    f"{MIN_STEPS_BEFORE_EARLY_STOP}"
)
print()
print(
    f"Config fingerprint       : "
    f"{CONFIG_FINGERPRINT[:16]}..."
)

✅ Architecture-A configuration loaded.

Run mode                 : production
Architecture             : Architecture-A-Decoder-Only-Transformer
Context length           : 2048
d_model                  : 768
Layers                   : 12
Heads                    : 12
Feed-forward dim         : 3072
Train token ceiling      : 100,000,000
Validation tokens        : 5,000,000
Tokens / optimizer step  : 16,384
Maximum optimizer steps  : 6,104

Early stopping
  Enabled                : True
  Min best checkpoints   : 8
  Max best checkpoints   : 12
  Patience               : 4
  Minimum meaningful Δ   : 0.005
  Minimum steps          : 2000

Config fingerprint       : 32b521bf882f1f23...


In [2]:
# Cell 02 — Initialize Architecture-A runtime, device, reproducibility, and model checks
# DECISION: UPDATE

import os
import random
import numpy as np
import torch

print("⚙️ Initializing Architecture-A runtime...")

# =========================================================================
# REPRODUCIBILITY
# =========================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================================================
# DEVICE SELECTION
# =========================================================================
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")

elif hasattr(torch, "backends") and hasattr(
    torch.backends,
    "mps",
) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

else:
    DEVICE = torch.device("cpu")

# =========================================================================
# CUDA PERFORMANCE SETTINGS
# =========================================================================
if DEVICE.type == "cuda":

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision(
            "high"
        )
    except Exception:
        pass

# =========================================================================
# MIXED PRECISION POLICY
# =========================================================================
if DEVICE.type == "cuda":

    if torch.cuda.is_bf16_supported():
        AMP_DTYPE = torch.bfloat16
        USE_AMP = True
        AMP_MODE = "bf16"

    else:
        AMP_DTYPE = torch.float16
        USE_AMP = True
        AMP_MODE = "fp16"

else:

    AMP_DTYPE = torch.float32
    USE_AMP = False
    AMP_MODE = "fp32"

# =========================================================================
# DEVICE INFORMATION
# =========================================================================
print()
print(f"Run mode             : {RUN_MODE}")
print(f"Device               : {DEVICE}")
print(f"AMP enabled          : {USE_AMP}")
print(f"AMP mode             : {AMP_MODE}")

if DEVICE.type == "cuda":
    print(
        f"GPU                  : "
        f"{torch.cuda.get_device_name(0)}"
    )

# =========================================================================
# MODEL CONFIGURATION SANITY CHECKS
# =========================================================================
assert (
    MODEL_CONFIG["d_model"]
    % MODEL_CONFIG["n_heads"]
    == 0
), (
    "d_model must be divisible by n_heads."
)

assert (
    MODEL_CONFIG["context_length"] > 0
)

assert (
    MODEL_CONFIG["vocab_size"] == 50_000
), (
    "Architecture-A must use the shared 50K tokenizer vocabulary."
)

assert (
    MODEL_CONFIG["pre_norm"] is True
)

assert (
    MODEL_CONFIG["position_type"]
    == "learned"
)

assert (
    MODEL_CONFIG["tie_embeddings"] is True
)

# =========================================================================
# TRAINING-POLICY SANITY CHECKS
# =========================================================================
assert (
    TRAIN_TOKENS > 0
)

assert (
    VAL_TOKENS > 0
)

assert (
    EVAL_EVERY_STEPS > 0
)

assert (
    CHECKPOINT_EVERY_STEPS > 0
)

assert (
    EARLY_STOPPING_PATIENCE >= 1
)

assert (
    EARLY_STOPPING_MIN_DELTA >= 0.0
)

assert (
    MIN_BEST_CHECKPOINTS_BEFORE_STOP
    >= 1
)

assert (
    MAX_BEST_CHECKPOINTS
    >= MIN_BEST_CHECKPOINTS_BEFORE_STOP
)

assert (
    MAX_STEPS >= 1
)

# =========================================================================
# FAST / PRODUCTION SUMMARY
# =========================================================================
print()
print("Training policy")
print(
    f"  Token ceiling       : "
    f"{TRAIN_TOKENS:,}"
)
print(
    f"  Maximum steps       : "
    f"{MAX_STEPS:,}"
)
print(
    f"  Validation interval : "
    f"{EVAL_EVERY_STEPS:,} steps"
)
print(
    f"  Checkpoint interval : "
    f"{CHECKPOINT_EVERY_STEPS:,} steps"
)
print(
    f"  Early stopping      : "
    f"{EARLY_STOPPING_ENABLED}"
)
print(
    f"  Min best checkpoints: "
    f"{MIN_BEST_CHECKPOINTS_BEFORE_STOP}"
)
print(
    f"  Max best checkpoints: "
    f"{MAX_BEST_CHECKPOINTS}"
)
print(
    f"  Patience            : "
    f"{EARLY_STOPPING_PATIENCE}"
)
print(
    f"  Minimum Δ loss      : "
    f"{EARLY_STOPPING_MIN_DELTA}"
)

print(
    "\n✅ Architecture-A runtime initialized."
)

⚙️ Initializing Architecture-A runtime...

Run mode             : production
Device               : cuda
AMP enabled          : True
AMP mode             : bf16
GPU                  : Tesla T4

Training policy
  Token ceiling       : 100,000,000
  Maximum steps       : 6,104
  Validation interval : 500 steps
  Checkpoint interval : 500 steps
  Early stopping      : True
  Min best checkpoints: 8
  Max best checkpoints: 12
  Patience            : 4
  Minimum Δ loss      : 0.005

✅ Architecture-A runtime initialized.


In [3]:
# Cell 03 — Resolve shared tokenizer and tokenized Phase-1 training shards
# DECISION: UPDATE

from pathlib import Path
import json
import numpy as np

print("📦 Resolving shared Phase-1 tokenizer and tokenized corpus...")

# =========================================================================
# EXPECTED SHARED DATASET STRUCTURE
# =========================================================================
EXPECTED_SHARD_NAMES = {
    "acts": "acts_train.bin",
    "high_courts": "high_courts_train.bin",
    "supreme_court": "supreme_court_train.bin",
    "tribunals": "tribunals_train.bin",
}

EXPECTED_VAL_SHARD_NAMES = {
    "acts": "acts_val.bin",
    "high_courts": "high_courts_val.bin",
    "supreme_court": "supreme_court_val.bin",
    "tribunals": "tribunals_val.bin",
}

# =========================================================================
# POSSIBLE ROOTS
# =========================================================================
CANDIDATE_ROOTS = [
    Path(
        "/kaggle/input/cs781-phase1-tokenized-corpus"
    ),
    Path(
        "/kaggle/input"
    ),
    Path(
        "/kaggle/working"
    ),
]

# Remove duplicates and non-existent roots.
SEARCH_ROOTS = []

for _root in CANDIDATE_ROOTS:
    if _root.exists():
        _resolved = _root.resolve()

        if _resolved not in SEARCH_ROOTS:
            SEARCH_ROOTS.append(
                _resolved
            )

assert SEARCH_ROOTS, (
    "No Kaggle data/working roots are available."
)

# =========================================================================
# TOKENIZER DISCOVERY
# =========================================================================
TOKENIZER_PATH = None

# Prefer the exact expected shared tokenizer path.
for _root in SEARCH_ROOTS:
    _candidate = (
        _root /
        "tokenizer" /
        "tokenizer.json"
    )

    if _candidate.is_file():
        TOKENIZER_PATH = _candidate
        break

# Fallback: search for tokenizer.json.
if TOKENIZER_PATH is None:

    _matches = []

    for _root in SEARCH_ROOTS:
        try:
            _matches.extend(
                _root.rglob("tokenizer.json")
            )
        except Exception:
            continue

    # Prefer paths belonging to the shared Phase-1 dataset.
    _matches = sorted(
        set(
            path.resolve()
            for path in _matches
            if path.is_file()
        ),
        key=lambda path: (
            "cs781-phase1-tokenized-corpus"
            not in str(path).lower(),
            len(str(path)),
        ),
    )

    if _matches:
        TOKENIZER_PATH = _matches[0]

assert TOKENIZER_PATH is not None, (
    "Shared tokenizer.json could not be located."
)

# =========================================================================
# LOAD TOKENIZER
# =========================================================================
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

TOKENIZER_VOCAB_SIZE = int(
    tokenizer.get_vocab_size()
)

assert (
    TOKENIZER_VOCAB_SIZE
    == MODEL_CONFIG["vocab_size"]
), (
    f"Tokenizer vocabulary size is "
    f"{TOKENIZER_VOCAB_SIZE}, but Architecture-A "
    f"expects {MODEL_CONFIG['vocab_size']}."
)

# =========================================================================
# DATASET ROOT DISCOVERY
# =========================================================================
TOKENIZED_ROOT = None

for _root in SEARCH_ROOTS:

    _candidate = (
        _root /
        "tokenized"
    )

    if _candidate.is_dir():

        _required = [
            _candidate /
            _filename
            for _filename in EXPECTED_SHARD_NAMES.values()
        ]

        if all(
            _path.is_file()
            for _path in _required
        ):
            TOKENIZED_ROOT = _candidate
            break

# Fallback search.
if TOKENIZED_ROOT is None:

    _candidate_dirs = []

    for _root in SEARCH_ROOTS:
        try:
            for _path in _root.rglob(
                "acts_train.bin"
            ):
                if _path.is_file():
                    _candidate_dirs.append(
                        _path.parent
                    )
        except Exception:
            continue

    _candidate_dirs = sorted(
        set(
            path.resolve()
            for path in _candidate_dirs
        ),
        key=lambda path: (
            "cs781-phase1-tokenized-corpus"
            not in str(path).lower(),
            len(str(path)),
        ),
    )

    for _directory in _candidate_dirs:

        _required = [
            _directory /
            _filename
            for _filename in EXPECTED_SHARD_NAMES.values()
        ]

        if all(
            _path.is_file()
            for _path in _required
        ):
            TOKENIZED_ROOT = _directory
            break

assert TOKENIZED_ROOT is not None, (
    "Could not locate the complete shared "
    "Phase-1 tokenized corpus."
)

# =========================================================================
# RESOLVE SHARD PATHS
# =========================================================================
TRAIN_SHARD_PATHS = {}

VAL_SHARD_PATHS = {}

for _source_name, _filename in (
    EXPECTED_SHARD_NAMES.items()
):

    _path = (
        TOKENIZED_ROOT /
        _filename
    )

    assert _path.is_file(), (
        f"Missing training shard: {_path}"
    )

    TRAIN_SHARD_PATHS[
        _source_name
    ] = _path

for _source_name, _filename in (
    EXPECTED_VAL_SHARD_NAMES.items()
):

    _path = (
        TOKENIZED_ROOT /
        _filename
    )

    assert _path.is_file(), (
        f"Missing validation shard: {_path}"
    )

    VAL_SHARD_PATHS[
        _source_name
    ] = _path

# =========================================================================
# TOKEN DTYPE
# =========================================================================
TOKEN_DTYPE = np.uint16

TOKEN_DTYPE_BYTES = (
    np.dtype(
        TOKEN_DTYPE
    ).itemsize
)

# =========================================================================
# SHARD METADATA
# =========================================================================
TRAIN_SHARD_TOKEN_COUNTS = {}

VAL_SHARD_TOKEN_COUNTS = {}

for _source_name, _path in (
    TRAIN_SHARD_PATHS.items()
):

    _count = (
        _path.stat().st_size
        // TOKEN_DTYPE_BYTES
    )

    assert _count > 0

    TRAIN_SHARD_TOKEN_COUNTS[
        _source_name
    ] = int(_count)

for _source_name, _path in (
    VAL_SHARD_PATHS.items()
):

    _count = (
        _path.stat().st_size
        // TOKEN_DTYPE_BYTES
    )

    assert _count > 0

    VAL_SHARD_TOKEN_COUNTS[
        _source_name
    ] = int(_count)

# =========================================================================
# TOTAL COUNTS
# =========================================================================
TOTAL_TRAIN_SHARD_TOKENS = int(
    sum(
        TRAIN_SHARD_TOKEN_COUNTS.values()
    )
)

TOTAL_VAL_SHARD_TOKENS = int(
    sum(
        VAL_SHARD_TOKEN_COUNTS.values()
    )
)

assert TOTAL_TRAIN_SHARD_TOKENS > 0
assert TOTAL_VAL_SHARD_TOKENS > 0

# =========================================================================
# OPTIONAL TOKENIZER METADATA
# =========================================================================
TOKENIZER_METADATA_PATH = (
    TOKENIZER_PATH.parent.parent /
    "metadata" /
    "tokenizer_metadata.json"
)

TOKENIZER_METADATA = {}

if TOKENIZER_METADATA_PATH.is_file():

    with open(
        TOKENIZER_METADATA_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        TOKENIZER_METADATA = json.load(f)

# =========================================================================
# SAVE RESOLVED DATA MANIFEST
# =========================================================================
DATA_MANIFEST = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "run_mode": RUN_MODE,
    "tokenizer_path": str(
        TOKENIZER_PATH
    ),
    "tokenizer_vocab_size": TOKENIZER_VOCAB_SIZE,
    "token_dtype": str(
        np.dtype(TOKEN_DTYPE)
    ),
    "tokenized_root": str(
        TOKENIZED_ROOT
    ),
    "train_shards": {
        name: {
            "path": str(path),
            "tokens": TRAIN_SHARD_TOKEN_COUNTS[name],
        }
        for name, path in TRAIN_SHARD_PATHS.items()
    },
    "validation_shards": {
        name: {
            "path": str(path),
            "tokens": VAL_SHARD_TOKEN_COUNTS[name],
        }
        for name, path in VAL_SHARD_PATHS.items()
    },
    "total_train_tokens_available": (
        TOTAL_TRAIN_SHARD_TOKENS
    ),
    "total_validation_tokens_available": (
        TOTAL_VAL_SHARD_TOKENS
    ),
}

DATA_MANIFEST_PATH = (
    CONFIGS_DIR /
    "architecture_A_data_manifest.json"
)

with open(
    DATA_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        DATA_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =========================================================================
# SUMMARY
# =========================================================================
print()
print("✅ Shared Phase-1 data resolved.")
print()
print(
    f"Tokenizer            : "
    f"{TOKENIZER_PATH}"
)
print(
    f"Vocabulary size      : "
    f"{TOKENIZER_VOCAB_SIZE:,}"
)
print(
    f"Token dtype          : "
    f"{np.dtype(TOKEN_DTYPE)}"
)
print(
    f"Train tokens available: "
    f"{TOTAL_TRAIN_SHARD_TOKENS:,}"
)
print(
    f"Val tokens available  : "
    f"{TOTAL_VAL_SHARD_TOKENS:,}"
)

print()
print("Training shards:")

for _name, _path in (
    TRAIN_SHARD_PATHS.items()
):

    print(
        f"  {_name:<16} "
        f"{TRAIN_SHARD_TOKEN_COUNTS[_name]:>15,} tokens"
    )

print()
print(
    f"Data manifest        : "
    f"{DATA_MANIFEST_PATH}"
)

📦 Resolving shared Phase-1 tokenizer and tokenized corpus...

✅ Shared Phase-1 data resolved.

Tokenizer            : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json
Vocabulary size      : 50,000
Token dtype          : uint16
Train tokens available: 4,335,607,302
Val tokens available  : 83,634,046

Training shards:
  acts                 149,598,391 tokens
  high_courts        2,327,101,667 tokens
  supreme_court        251,833,809 tokens
  tribunals          1,607,073,435 tokens

Data manifest        : /kaggle/working/architecture_A/configs/architecture_A_data_manifest.json


In [4]:
# Cell 04 — Build memory-mapped training/validation samplers for Architecture-A
# DECISION: UPDATE

import numpy as np
import torch

print("🎯 Building Architecture-A data samplers...")

# =========================================================================
# SOURCE MIX
# =========================================================================
SOURCE_NAMES = (
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
)

SOURCE_PROBABILITIES = {
    "acts": 0.25,
    "high_courts": 0.25,
    "supreme_court": 0.25,
    "tribunals": 0.25,
}

assert set(
    SOURCE_PROBABILITIES.keys()
) == set(
    SOURCE_NAMES
)

assert abs(
    sum(SOURCE_PROBABILITIES.values()) - 1.0
) < 1e-8

# =========================================================================
# MEMORY-MAPPED SHARDS
# =========================================================================
# IMPORTANT:
# np.memmap does NOT load the complete corpus into RAM.
# It only creates a file-backed view.
TRAIN_SHARDS = {
    _name: np.memmap(
        _path,
        dtype=TOKEN_DTYPE,
        mode="r",
    )
    for _name, _path in TRAIN_SHARD_PATHS.items()
}

VAL_SHARDS = {
    _name: np.memmap(
        _path,
        dtype=TOKEN_DTYPE,
        mode="r",
    )
    for _name, _path in VAL_SHARD_PATHS.items()
}

# =========================================================================
# FAST SHARD VALIDATION
# =========================================================================
# Do NOT perform np.min()/np.max() over the full memmap.
# That would scan potentially billions of tokens.
#
# Instead, inspect a handful of positions from each shard.
# =========================================================================
PROBE_POSITIONS = (
    0,
    1,
    2,
    10,
    100,
    -1,
)

for _name in SOURCE_NAMES:

    train_shard = TRAIN_SHARDS[
        _name
    ]

    val_shard = VAL_SHARDS[
        _name
    ]

    assert len(
        train_shard
    ) == TRAIN_SHARD_TOKEN_COUNTS[
        _name
    ]

    assert len(
        val_shard
    ) == VAL_SHARD_TOKEN_COUNTS[
        _name
    ]

    assert (
        len(train_shard)
        > TRAIN_CONTEXT_LENGTH
        if "TRAIN_CONTEXT_LENGTH" in globals()
        else len(train_shard)
        > MODEL_CONFIG["context_length"]
    )

    assert (
        len(val_shard)
        > TRAIN_CONTEXT_LENGTH
        if "TRAIN_CONTEXT_LENGTH" in globals()
        else len(val_shard)
        > MODEL_CONFIG["context_length"]
    )

    _context = int(
        MODEL_CONFIG["context_length"]
    )

    for _position in PROBE_POSITIONS:

        if (
            _position >= len(train_shard)
            or abs(_position) > len(train_shard)
        ):
            continue

        _train_value = int(
            train_shard[
                _position
            ]
        )

        assert (
            0
            <= _train_value
            < MODEL_CONFIG["vocab_size"]
        ), (
            f"Invalid token ID in "
            f"{_name} training shard: "
            f"{_train_value}"
        )

        if (
            _position >= len(val_shard)
            or abs(_position) > len(val_shard)
        ):
            continue

        _val_value = int(
            val_shard[
                _position
            ]
        )

        assert (
            0
            <= _val_value
            < MODEL_CONFIG["vocab_size"]
        ), (
            f"Invalid token ID in "
            f"{_name} validation shard: "
            f"{_val_value}"
        )

# =========================================================================
# WINDOW SAMPLER
# =========================================================================
class CausalTokenSampler:

    def __init__(
        self,
        shards,
        source_probabilities,
        context_length,
        seed,
    ):
        self.shards = shards

        self.source_names = tuple(
            source_probabilities.keys()
        )

        self.source_probabilities = np.asarray(
            [
                source_probabilities[
                    name
                ]
                for name in self.source_names
            ],
            dtype=np.float64,
        )

        self.source_probabilities /= (
            self.source_probabilities.sum()
        )

        self.context_length = int(
            context_length
        )

        self.seed = int(
            seed
        )

        self.rng = np.random.default_rng(
            self.seed
        )

        self.sample_count = 0

        for _name in self.source_names:

            assert len(
                self.shards[_name]
            ) > (
                self.context_length + 1
            ), (
                f"Shard {_name} is too short for "
                f"context_length="
                f"{self.context_length}."
            )

    # ---------------------------------------------------------------------
    # Sample source according to configured mixture.
    # ---------------------------------------------------------------------
    def _sample_source(self):

        _index = int(
            self.rng.choice(
                len(self.source_names),
                p=self.source_probabilities,
            )
        )

        return self.source_names[
            _index
        ]

    # ---------------------------------------------------------------------
    # Sample one next-token batch.
    # ---------------------------------------------------------------------
    def sample_batch(
        self,
        batch_size,
        device,
    ):

        batch_size = int(
            batch_size
        )

        assert batch_size > 0

        x = np.empty(
            (
                batch_size,
                self.context_length,
            ),
            dtype=np.int64,
        )

        y = np.empty(
            (
                batch_size,
                self.context_length,
            ),
            dtype=np.int64,
        )

        sampled_sources = []

        for _row in range(
            batch_size
        ):

            source_name = (
                self._sample_source()
            )

            shard = self.shards[
                source_name
            ]

            max_start = (
                len(shard)
                - self.context_length
                - 1
            )

            start = int(
                self.rng.integers(
                    0,
                    max_start + 1,
                )
            )

            window = np.asarray(
                shard[
                    start:
                    start
                    + self.context_length
                    + 1
                ],
                dtype=np.int64,
            )

            x[
                _row
            ] = window[:-1]

            y[
                _row
            ] = window[1:]

            sampled_sources.append(
                source_name
            )

        self.sample_count += (
            batch_size
        )

        return (
            torch.from_numpy(
                x
            ).to(
                device=device,
                non_blocking=True,
            ),
            torch.from_numpy(
                y
            ).to(
                device=device,
                non_blocking=True,
            ),
            sampled_sources,
        )

    # ---------------------------------------------------------------------
    # Checkpointable state.
    # ---------------------------------------------------------------------
    def state_dict(self):

        return {
            "seed": int(
                self.seed
            ),
            "sample_count": int(
                self.sample_count
            ),
            "rng_state": (
                self.rng.bit_generator.state
            ),
            "context_length": int(
                self.context_length
            ),
            "source_names": list(
                self.source_names
            ),
            "source_probabilities": (
                self.source_probabilities.tolist()
            ),
        }

    # ---------------------------------------------------------------------
    # Restore state.
    # ---------------------------------------------------------------------
    def load_state_dict(
        self,
        state,
    ):

        assert int(
            state["context_length"]
        ) == self.context_length

        assert tuple(
            state["source_names"]
        ) == self.source_names

        np.testing.assert_allclose(
            np.asarray(
                state[
                    "source_probabilities"
                ],
                dtype=np.float64,
            ),
            self.source_probabilities,
            rtol=0.0,
            atol=1e-12,
        )

        self.seed = int(
            state["seed"]
        )

        self.sample_count = int(
            state["sample_count"]
        )

        self.rng.bit_generator.state = (
            state["rng_state"]
        )


# =========================================================================
# TRAINING / VALIDATION CONTEXT
# =========================================================================
TRAIN_CONTEXT_LENGTH = int(
    MODEL_CONFIG["context_length"]
)

assert TRAIN_CONTEXT_LENGTH > 0

# =========================================================================
# VALIDATION SETTINGS
# =========================================================================
if RUN_MODE == "fast":

    VAL_BATCH_SIZE = 2
    VAL_EVAL_BATCHES = 4

else:

    VAL_BATCH_SIZE = 1
    VAL_EVAL_BATCHES = 100

# =========================================================================
# CREATE SAMPLERS
# =========================================================================
TRAIN_SAMPLER = CausalTokenSampler(
    shards=TRAIN_SHARDS,
    source_probabilities=SOURCE_PROBABILITIES,
    context_length=TRAIN_CONTEXT_LENGTH,
    seed=SEED + 1_000,
)

VAL_SAMPLER = CausalTokenSampler(
    shards=VAL_SHARDS,
    source_probabilities=SOURCE_PROBABILITIES,
    context_length=TRAIN_CONTEXT_LENGTH,
    seed=SEED + 10_000,
)

# =========================================================================
# SMALL SAMPLER PROBE
# =========================================================================
# Only one actual training window is sampled here.
# The sampler RNG is then reset so this probe has no effect on training.
_probe_x, _probe_y, _probe_sources = (
    TRAIN_SAMPLER.sample_batch(
        batch_size=1,
        device=DEVICE,
    )
)

assert _probe_x.shape == (
    1,
    TRAIN_CONTEXT_LENGTH,
)

assert _probe_y.shape == (
    1,
    TRAIN_CONTEXT_LENGTH,
)

assert _probe_x.dtype == torch.long
assert _probe_y.dtype == torch.long

# Reset sampler after probe.
TRAIN_SAMPLER = CausalTokenSampler(
    shards=TRAIN_SHARDS,
    source_probabilities=SOURCE_PROBABILITIES,
    context_length=TRAIN_CONTEXT_LENGTH,
    seed=SEED + 1_000,
)

VAL_SAMPLER = CausalTokenSampler(
    shards=VAL_SHARDS,
    source_probabilities=SOURCE_PROBABILITIES,
    context_length=TRAIN_CONTEXT_LENGTH,
    seed=SEED + 10_000,
)

# =========================================================================
# SUMMARY
# =========================================================================
print()
print("✅ Architecture-A samplers ready.")
print()
print(
    f"Context length        : "
    f"{TRAIN_CONTEXT_LENGTH}"
)

print(
    f"Train batch size      : "
    f"{MICRO_BATCH_SIZE}"
)

print(
    f"Gradient accumulation : "
    f"{GRAD_ACCUM_STEPS}"
)

print(
    f"Validation batch size : "
    f"{VAL_BATCH_SIZE}"
)

print(
    f"Validation batches    : "
    f"{VAL_EVAL_BATCHES}"
)

print()
print("Source probabilities:")

for _name in SOURCE_NAMES:
    print(
        f"  {_name:<16} "
        f"{SOURCE_PROBABILITIES[_name]:.2f}"
    )

print()
print(
    f"Train sampler seed    : "
    f"{SEED + 1_000}"
)

print(
    f"Validation sampler seed: "
    f"{SEED + 10_000}"
)

🎯 Building Architecture-A data samplers...

✅ Architecture-A samplers ready.

Context length        : 2048
Train batch size      : 1
Gradient accumulation : 8
Validation batch size : 1
Validation batches    : 100

Source probabilities:
  acts             0.25
  high_courts      0.25
  supreme_court    0.25
  tribunals        0.25

Train sampler seed    : 1042
Validation sampler seed: 10042


In [5]:
# Cell 05 — Build Architecture-A decoder-only Transformer
# DECISION: UPDATE

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print("🏗️ Building Architecture-A decoder-only Transformer...")

# =========================================================================
# CAUSAL SELF-ATTENTION
# =========================================================================
class CausalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout,
        max_seq_len,
        use_bias=True,
    ):
        super().__init__()

        assert d_model % n_heads == 0, (
            "d_model must be divisible by n_heads."
        )

        self.d_model = int(d_model)
        self.n_heads = int(n_heads)
        self.d_head = (
            self.d_model // self.n_heads
        )
        self.max_seq_len = int(
            max_seq_len
        )

        self.q_proj = nn.Linear(
            d_model,
            d_model,
            bias=use_bias,
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model,
            bias=use_bias,
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model,
            bias=use_bias,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=use_bias,
        )

        self.attn_dropout = nn.Dropout(
            dropout
        )

        self.resid_dropout = nn.Dropout(
            dropout
        )

        # -----------------------------------------------------------------
        # Causal mask.
        #
        # True  -> attention is allowed.
        # False -> future position is blocked.
        # -----------------------------------------------------------------
        causal_mask = torch.tril(
            torch.ones(
                max_seq_len,
                max_seq_len,
                dtype=torch.bool,
            )
        )

        self.register_buffer(
            "causal_mask",
            causal_mask,
            persistent=False,
        )

        self.attention_scale = (
            1.0 / math.sqrt(
                self.d_head
            )
        )

    def forward(
        self,
        x,
    ):

        B, T, C = x.shape

        assert C == self.d_model

        assert T <= (
            self.max_seq_len
        ), (
            f"Sequence length {T} exceeds "
            f"maximum {self.max_seq_len}."
        )

        # -------------------------------------------------------------
        # Project queries, keys and values.
        # -------------------------------------------------------------
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # -------------------------------------------------------------
        # [B,T,C] -> [B,H,T,D]
        # -------------------------------------------------------------
        q = q.view(
            B,
            T,
            self.n_heads,
            self.d_head,
        ).transpose(
            1,
            2,
        )

        k = k.view(
            B,
            T,
            self.n_heads,
            self.d_head,
        ).transpose(
            1,
            2,
        )

        v = v.view(
            B,
            T,
            self.n_heads,
            self.d_head,
        ).transpose(
            1,
            2,
        )

        # -------------------------------------------------------------
        # Attention scores.
        # -------------------------------------------------------------
        scores = (
            q
            @ k.transpose(
                -2,
                -1,
            )
        ) * self.attention_scale

        # -------------------------------------------------------------
        # Causal masking.
        # -------------------------------------------------------------
        mask = self.causal_mask[
            :T,
            :T,
        ]

        scores = scores.masked_fill(
            ~mask.view(
                1,
                1,
                T,
                T,
            ),
            float("-inf"),
        )

        # -------------------------------------------------------------
        # Normalize attention.
        # -------------------------------------------------------------
        attn = F.softmax(
            scores.float(),
            dim=-1,
        ).to(
            scores.dtype
        )

        attn = self.attn_dropout(
            attn
        )

        # -------------------------------------------------------------
        # Weighted value aggregation.
        # -------------------------------------------------------------
        out = (
            attn @ v
        )

        # [B,H,T,D] -> [B,T,C]
        out = (
            out.transpose(
                1,
                2,
            )
            .contiguous()
            .view(
                B,
                T,
                C,
            )
        )

        out = self.out_proj(
            out
        )

        out = self.resid_dropout(
            out
        )

        return out


# =========================================================================
# GELU FEED-FORWARD NETWORK
# =========================================================================
class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout,
        use_bias=True,
    ):
        super().__init__()

        self.fc1 = nn.Linear(
            d_model,
            d_ff,
            bias=use_bias,
        )

        self.fc2 = nn.Linear(
            d_ff,
            d_model,
            bias=use_bias,
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
    ):

        x = self.fc1(
            x
        )

        x = F.gelu(
            x,
            approximate="none",
        )

        x = self.fc2(
            x
        )

        x = self.dropout(
            x
        )

        return x


# =========================================================================
# PRE-LN TRANSFORMER BLOCK
# =========================================================================
class TransformerBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout,
        max_seq_len,
        use_bias=True,
    ):
        super().__init__()

        self.ln_1 = nn.LayerNorm(
            d_model
        )

        self.attention = (
            CausalSelfAttention(
                d_model=d_model,
                n_heads=n_heads,
                dropout=dropout,
                max_seq_len=max_seq_len,
                use_bias=use_bias,
            )
        )

        self.ln_2 = nn.LayerNorm(
            d_model
        )

        self.feed_forward = (
            FeedForward(
                d_model=d_model,
                d_ff=d_ff,
                dropout=dropout,
                use_bias=use_bias,
            )
        )

    def forward(
        self,
        x,
    ):

        # -------------------------------------------------------------
        # Pre-LN attention residual.
        # -------------------------------------------------------------
        x = x + self.attention(
            self.ln_1(x)
        )

        # -------------------------------------------------------------
        # Pre-LN feed-forward residual.
        # -------------------------------------------------------------
        x = x + self.feed_forward(
            self.ln_2(x)
        )

        return x


# =========================================================================
# COMPLETE DECODER-ONLY LANGUAGE MODEL
# =========================================================================
class DecoderOnlyTransformerLM(nn.Module):

    def __init__(
        self,
        config,
    ):
        super().__init__()

        self.config = dict(
            config
        )

        vocab_size = int(
            config["vocab_size"]
        )

        d_model = int(
            config["d_model"]
        )

        n_layers = int(
            config["n_layers"]
        )

        n_heads = int(
            config["n_heads"]
        )

        d_ff = int(
            config["d_ff"]
        )

        dropout = float(
            config["dropout"]
        )

        max_seq_len = int(
            config["context_length"]
        )

        use_bias = bool(
            config["use_bias"]
        )

        # -------------------------------------------------------------
        # Token embeddings.
        # -------------------------------------------------------------
        self.token_embedding = (
            nn.Embedding(
                vocab_size,
                d_model,
            )
        )

        # -------------------------------------------------------------
        # Learned positional embeddings.
        # -------------------------------------------------------------
        self.position_embedding = (
            nn.Embedding(
                max_seq_len,
                d_model,
            )
        )

        self.embedding_dropout = (
            nn.Dropout(
                dropout
            )
        )

        # -------------------------------------------------------------
        # Decoder blocks.
        # -------------------------------------------------------------
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    d_model=d_model,
                    n_heads=n_heads,
                    d_ff=d_ff,
                    dropout=dropout,
                    max_seq_len=max_seq_len,
                    use_bias=use_bias,
                )
                for _ in range(
                    n_layers
                )
            ]
        )

        # -------------------------------------------------------------
        # Final normalization.
        # -------------------------------------------------------------
        self.final_norm = nn.LayerNorm(
            d_model
        )

        # -------------------------------------------------------------
        # Weight tying:
        # output projection uses token_embedding.weight directly.
        #
        # Therefore there is no second 50K x d_model parameter matrix.
        # -------------------------------------------------------------
        self.tie_embeddings = bool(
            config["tie_embeddings"]
        )

        if not self.tie_embeddings:

            self.lm_head = nn.Linear(
                d_model,
                vocab_size,
                bias=False,
            )

        else:

            self.lm_head = None

        # -------------------------------------------------------------
        # Initialize weights.
        # -------------------------------------------------------------
        self.apply(
            self._init_weights
        )

        # -------------------------------------------------------------
        # Explicitly tie output weights if requested.
        # -------------------------------------------------------------
        if self.tie_embeddings:

            # No second parameter tensor is created.
            self.output_weight = (
                self.token_embedding.weight
            )

    def _init_weights(
        self,
        module,
    ):

        if isinstance(
            module,
            nn.Linear,
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding,
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

        elif isinstance(
            module,
            nn.LayerNorm,
        ):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )

    def forward(
        self,
        input_ids,
        targets=None,
    ):

        B, T = input_ids.shape

        assert T <= int(
            self.config[
                "context_length"
            ]
        ), (
            f"Sequence length {T} exceeds "
            f"context length "
            f"{self.config['context_length']}."
        )

        # -------------------------------------------------------------
        # Position indices.
        # -------------------------------------------------------------
        positions = torch.arange(
            T,
            device=input_ids.device,
            dtype=torch.long,
        )

        positions = positions.unsqueeze(
            0
        )

        # -------------------------------------------------------------
        # Token + position embeddings.
        # -------------------------------------------------------------
        x = (
            self.token_embedding(
                input_ids
            )
            + self.position_embedding(
                positions
            )
        )

        x = self.embedding_dropout(
            x
        )

        # -------------------------------------------------------------
        # Decoder stack.
        # -------------------------------------------------------------
        for block in self.blocks:

            x = block(
                x
            )

        # -------------------------------------------------------------
        # Final normalization.
        # -------------------------------------------------------------
        x = self.final_norm(
            x
        )

        # -------------------------------------------------------------
        # Tied or untied language-model head.
        # -------------------------------------------------------------
        if self.tie_embeddings:

            logits = F.linear(
                x,
                self.token_embedding.weight,
            )

        else:

            logits = self.lm_head(
                x
            )

        # -------------------------------------------------------------
        # Training loss.
        # -------------------------------------------------------------
        loss = None

        if targets is not None:

            # Float32 loss calculation improves numerical stability
            # under BF16/FP16 mixed precision.
            logits_for_loss = (
                logits.float()
            )

            loss = F.cross_entropy(
                logits_for_loss.view(
                    -1,
                    logits_for_loss.size(
                        -1
                    ),
                ),
                targets.reshape(
                    -1
                ),
            )

        return logits, loss


# =========================================================================
# INSTANTIATE MODEL
# =========================================================================
MODEL = DecoderOnlyTransformerLM(
    MODEL_CONFIG
).to(
    DEVICE
)

# =========================================================================
# PARAMETER COUNT
# =========================================================================
def count_trainable_parameters(
    model,
):
    return int(
        sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )
    )


PARAMETER_COUNT = (
    count_trainable_parameters(
        MODEL
    )
)

PARAMETER_MILLIONS = (
    PARAMETER_COUNT / 1_000_000
)

# =========================================================================
# PHASE-1 PARAMETER CONSTRAINT
# =========================================================================
if RUN_MODE == "production":

    assert (
        50_000_000
        <= PARAMETER_COUNT
        <= 300_000_000
    ), (
        f"Architecture-A production model has "
        f"{PARAMETER_COUNT:,} parameters, "
        "outside the Phase-1 50M–300M range."
    )

# =========================================================================
# FAST / PRODUCTION MODEL SUMMARY
# =========================================================================
print()
print("✅ Architecture-A model created.")
print()
print(
    f"Run mode             : "
    f"{RUN_MODE}"
)

print(
    f"Model type           : "
    f"decoder-only Transformer"
)

print(
    f"Vocabulary           : "
    f"{MODEL_CONFIG['vocab_size']:,}"
)

print(
    f"Context length       : "
    f"{MODEL_CONFIG['context_length']}"
)

print(
    f"d_model              : "
    f"{MODEL_CONFIG['d_model']}"
)

print(
    f"Layers               : "
    f"{MODEL_CONFIG['n_layers']}"
)

print(
    f"Heads                : "
    f"{MODEL_CONFIG['n_heads']}"
)

print(
    f"FFN dimension        : "
    f"{MODEL_CONFIG['d_ff']}"
)

print(
    f"Position encoding    : "
    f"{MODEL_CONFIG['position_type']}"
)

print(
    f"Normalization        : "
    f"Pre-LayerNorm"
)

print(
    f"Attention            : "
    f"Causal self-attention"
)

print(
    f"Activation           : "
    f"{MODEL_CONFIG['activation']}"
)

print(
    f"Weight tying         : "
    f"{MODEL_CONFIG['tie_embeddings']}"
)

print(
    f"Trainable parameters : "
    f"{PARAMETER_COUNT:,}"
)

print(
    f"Parameters (M)       : "
    f"{PARAMETER_MILLIONS:.2f}"
)

🏗️ Building Architecture-A decoder-only Transformer...

✅ Architecture-A model created.

Run mode             : production
Model type           : decoder-only Transformer
Vocabulary           : 50,000
Context length       : 2048
d_model              : 768
Layers               : 12
Heads                : 12
FFN dimension        : 3072
Position encoding    : learned
Normalization        : Pre-LayerNorm
Attention            : Causal self-attention
Activation           : gelu
Weight tying         : True
Trainable parameters : 125,028,864
Parameters (M)       : 125.03


In [6]:
# Cell 06 — Configure Architecture-A optimizer, scheduler, and precision policy
# DECISION: UPDATE

import math
import json
import torch

print("⚙️ Configuring Architecture-A optimization...")

# =========================================================================
# BASE OPTIMIZATION VALUES
# =========================================================================
BASE_LR = float(
    OPTIMIZER_CONFIG["lr"]
)

MIN_LR = float(
    OPTIMIZER_CONFIG["min_lr"]
)

WEIGHT_DECAY = float(
    OPTIMIZER_CONFIG["weight_decay"]
)

GRAD_CLIP_NORM = float(
    OPTIMIZER_CONFIG["grad_clip_norm"]
)

BETA1, BETA2 = tuple(
    OPTIMIZER_CONFIG["betas"]
)

assert BASE_LR > 0.0
assert MIN_LR > 0.0
assert MIN_LR <= BASE_LR
assert WEIGHT_DECAY >= 0.0
assert GRAD_CLIP_NORM > 0.0

# =========================================================================
# PARAMETER GROUPING
# =========================================================================
DECAY_PARAMETERS = []
NO_DECAY_PARAMETERS = []

for _name, _parameter in MODEL.named_parameters():

    if not _parameter.requires_grad:
        continue

    # Decay matrix-like parameters.
    # Do not decay bias vectors or normalization parameters.
    if (
        _parameter.ndim >= 2
        and "norm" not in _name.lower()
    ):
        DECAY_PARAMETERS.append(
            _parameter
        )

    else:
        NO_DECAY_PARAMETERS.append(
            _parameter
        )

# -------------------------------------------------------------------------
# Verify every trainable parameter occurs exactly once.
# -------------------------------------------------------------------------
_DECAY_IDS = {
    id(_parameter)
    for _parameter in DECAY_PARAMETERS
}

_NO_DECAY_IDS = {
    id(_parameter)
    for _parameter in NO_DECAY_PARAMETERS
}

_ALL_TRAINABLE_IDS = {
    id(_parameter)
    for _parameter in MODEL.parameters()
    if _parameter.requires_grad
}

assert _DECAY_IDS.isdisjoint(
    _NO_DECAY_IDS
)

assert (
    _DECAY_IDS
    | _NO_DECAY_IDS
) == _ALL_TRAINABLE_IDS

# =========================================================================
# ADAMW
# =========================================================================
OPTIMIZER = torch.optim.AdamW(
    [
        {
            "params": DECAY_PARAMETERS,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": NO_DECAY_PARAMETERS,
            "weight_decay": 0.0,
        },
    ],
    lr=BASE_LR,
    betas=(BETA1, BETA2),
    eps=1e-8,
)

# =========================================================================
# LEARNING-RATE SCHEDULE
# =========================================================================
def learning_rate_multiplier(
    step
):
    """
    Linear warmup followed by cosine decay.

    Returns a multiplier relative to BASE_LR.
    """

    step = int(step)

    # -------------------------------------------------------------
    # Linear warmup.
    # -------------------------------------------------------------
    if WARMUP_STEPS > 0:

        if step < WARMUP_STEPS:

            return float(
                step + 1
            ) / float(
                WARMUP_STEPS
            )

    # -------------------------------------------------------------
    # No meaningful decay region.
    # -------------------------------------------------------------
    if MAX_STEPS <= WARMUP_STEPS:

        return 1.0

    # -------------------------------------------------------------
    # Cosine decay.
    # -------------------------------------------------------------
    progress = (
        float(
            step - WARMUP_STEPS
        )
        / float(
            MAX_STEPS - WARMUP_STEPS
        )
    )

    progress = min(
        1.0,
        max(
            0.0,
            progress,
        ),
    )

    cosine = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * progress
            )
        )
    )

    minimum_multiplier = (
        MIN_LR / BASE_LR
    )

    return float(
        minimum_multiplier
        + (
            1.0
            - minimum_multiplier
        )
        * cosine
    )


LR_SCHEDULER = (
    torch.optim.lr_scheduler.LambdaLR(
        OPTIMIZER,
        lr_lambda=learning_rate_multiplier,
    )
)

# =========================================================================
# CURRENT LEARNING RATE
# =========================================================================
CURRENT_LR = float(
    OPTIMIZER.param_groups[0]["lr"]
)

assert CURRENT_LR > 0.0
assert CURRENT_LR <= BASE_LR + 1e-12

# =========================================================================
# PRECISION / GRADIENT SCALER
# =========================================================================
if (
    DEVICE.type == "cuda"
    and AMP_MODE == "fp16"
):

    GRAD_SCALER = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )

else:

    GRAD_SCALER = torch.amp.GradScaler(
        "cuda",
        enabled=False,
    )

# =========================================================================
# AUTOCast HELPER
# =========================================================================
def autocast_context():

    if (
        DEVICE.type == "cuda"
        and USE_AMP
    ):

        return torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=True,
        )

    # Disabled autocast context for MPS / CPU.
    return torch.autocast(
        device_type="cpu",
        dtype=torch.float32,
        enabled=False,
    )

# =========================================================================
# LR INSPECTION HELPER
# =========================================================================
def get_current_learning_rate():

    return float(
        OPTIMIZER.param_groups[0]["lr"]
    )

# =========================================================================
# SAVE OPTIMIZATION MANIFEST
# =========================================================================
OPTIMIZATION_MANIFEST = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "run_mode": RUN_MODE,

    "optimizer": OPTIMIZER_NAME,

    "base_learning_rate": BASE_LR,
    "minimum_learning_rate": MIN_LR,

    "current_learning_rate": (
        get_current_learning_rate()
    ),

    "betas": [
        float(BETA1),
        float(BETA2),
    ],

    "weight_decay": WEIGHT_DECAY,

    "gradient_clip_norm": (
        GRAD_CLIP_NORM
    ),

    "warmup_steps": int(
        WARMUP_STEPS
    ),

    "maximum_steps": int(
        MAX_STEPS
    ),

    "schedule": (
        "linear_warmup_cosine_decay"
    ),

    "amp_mode": AMP_MODE,

    "use_amp": bool(
        USE_AMP
    ),
}

OPTIMIZATION_MANIFEST_PATH = (
    CONFIGS_DIR
    / "architecture_A_optimization_manifest.json"
)

with open(
    OPTIMIZATION_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        OPTIMIZATION_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =========================================================================
# SUMMARY
# =========================================================================
print()
print("✅ Architecture-A optimization configured.")
print()

print(
    f"Optimizer             : "
    f"{OPTIMIZER_NAME}"
)

print(
    f"Base learning rate    : "
    f"{BASE_LR:.6g}"
)

print(
    f"Current learning rate : "
    f"{CURRENT_LR:.6g}"
)

print(
    f"Minimum learning rate : "
    f"{MIN_LR:.6g}"
)

print(
    f"Weight decay          : "
    f"{WEIGHT_DECAY}"
)

print(
    f"Gradient clipping     : "
    f"{GRAD_CLIP_NORM}"
)

print(
    f"Warmup steps          : "
    f"{WARMUP_STEPS:,}"
)

print(
    f"Maximum steps         : "
    f"{MAX_STEPS:,}"
)

print(
    f"Precision             : "
    f"{AMP_MODE}"
)

print(
    f"AMP enabled           : "
    f"{USE_AMP}"
)

print(
    f"Decay tensors         : "
    f"{len(DECAY_PARAMETERS):,}"
)

print(
    f"No-decay tensors      : "
    f"{len(NO_DECAY_PARAMETERS):,}"
)

print(
    f"Optimization manifest : "
    f"{OPTIMIZATION_MANIFEST_PATH}"
)

⚙️ Configuring Architecture-A optimization...

✅ Architecture-A optimization configured.

Optimizer             : AdamW
Base learning rate    : 0.0003
Current learning rate : 9.83607e-07
Minimum learning rate : 3e-05
Weight decay          : 0.1
Gradient clipping     : 1.0
Warmup steps          : 305
Maximum steps         : 6,104
Precision             : bf16
AMP enabled           : True
Decay tensors         : 74
No-decay tensors      : 122
Optimization manifest : /kaggle/working/architecture_A/configs/architecture_A_optimization_manifest.json


In [7]:
# Cell 07 — Build Architecture-A checkpoint, resume, and compatibility infrastructure
# DECISION: UPDATE

import os
import json
import random
from pathlib import Path

import numpy as np
import torch

print("💾 Building Architecture-A checkpoint infrastructure...")

# =========================================================================
# CHECKPOINT PATHS
# =========================================================================
LATEST_CHECKPOINT_PATH = CHECKPOINTS_DIR / "latest.pt"
BEST_CHECKPOINT_PATH = CHECKPOINTS_DIR / "best.pt"
CHECKPOINT_INDEX_PATH = (
    METRICS_DIR / "checkpoint_index_A.json"
)

CHECKPOINT_SCHEMA_VERSION = 2

# =========================================================================
# CONFIGURATION FINGERPRINT
# =========================================================================
def get_checkpoint_signature():

    return {
        "experiment_id": EXPERIMENT_ID,
        "architecture_name": ARCHITECTURE_NAME,
        "architecture_version": ARCHITECTURE_VERSION,
        "run_mode": RUN_MODE,
        "config_fingerprint": CONFIG_FINGERPRINT,
    }


CHECKPOINT_SIGNATURE = (
    get_checkpoint_signature()
)

# =========================================================================
# COMPATIBILITY CHECK
# =========================================================================
def checkpoint_is_compatible(checkpoint):

    if not isinstance(checkpoint, dict):
        return False

    if checkpoint.get(
        "checkpoint_schema_version"
    ) != CHECKPOINT_SCHEMA_VERSION:
        return False

    for _key, _value in (
        CHECKPOINT_SIGNATURE.items()
    ):
        if checkpoint.get(_key) != _value:
            return False

    return True


# =========================================================================
# RNG STATE
# =========================================================================
def capture_rng_state():

    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": None,
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = (
            torch.cuda.get_rng_state_all()
        )

    return state


def restore_rng_state(state):

    random.setstate(
        state["python"]
    )

    np.random.set_state(
        state["numpy"]
    )

    torch.set_rng_state(
        state["torch_cpu"]
    )

    if (
        torch.cuda.is_available()
        and state.get("torch_cuda") is not None
    ):
        torch.cuda.set_rng_state_all(
            state["torch_cuda"]
        )


# =========================================================================
# ATOMIC SAVE
# =========================================================================
def atomic_torch_save(
    obj,
    path,
):

    path = Path(path)

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        obj,
        temp_path,
    )

    os.replace(
        temp_path,
        path,
    )


# =========================================================================
# BUILD CHECKPOINT
# =========================================================================
def build_checkpoint(
    step,
    tokens_seen,
    train_loss,
    best_validation_loss,
    best_validation_step,
    bad_validation_count,
    best_checkpoint_count,
    history,
    status,
):

    return {
        "checkpoint_schema_version": (
            CHECKPOINT_SCHEMA_VERSION
        ),

        **CHECKPOINT_SIGNATURE,

        # Model
        "model": MODEL.state_dict(),

        # Optimizer / scheduler / precision
        "optimizer": OPTIMIZER.state_dict(),
        "scheduler": LR_SCHEDULER.state_dict(),
        "scaler": GRAD_SCALER.state_dict(),

        "current_lr": (
            get_current_learning_rate()
        ),

        # Data state
        "train_sampler": (
            TRAIN_SAMPLER.state_dict()
        ),
        "val_sampler": (
            VAL_SAMPLER.state_dict()
        ),

        # Progress
        "step": int(step),
        "tokens_seen": int(tokens_seen),
        "train_loss": float(train_loss),

        # Early stopping
        "best_validation_loss": float(
            best_validation_loss
        ),
        "best_validation_step": int(
            best_validation_step
        ),
        "bad_validation_count": int(
            bad_validation_count
        ),
        "best_checkpoint_count": int(
            best_checkpoint_count
        ),

        # History
        "history": history,

        # Status
        "status": str(status),

        # RNG
        "rng_state": (
            capture_rng_state()
        ),
    }


# =========================================================================
# CHECKPOINT INDEX
# =========================================================================
def create_checkpoint_index():

    return {
        **CHECKPOINT_SIGNATURE,

        "best_checkpoints": [],

        "latest_checkpoint": None,
    }


def load_checkpoint_index():

    if not CHECKPOINT_INDEX_PATH.is_file():
        return create_checkpoint_index()

    try:

        with open(
            CHECKPOINT_INDEX_PATH,
            "r",
            encoding="utf-8",
        ) as f:

            index = json.load(f)

    except Exception:

        return create_checkpoint_index()

    for _key, _value in (
        CHECKPOINT_SIGNATURE.items()
    ):

        if index.get(_key) != _value:

            print(
                "⚠️ Existing checkpoint index "
                "belongs to another configuration."
            )

            return create_checkpoint_index()

    return index


def save_checkpoint_index(index):

    temp_path = (
        CHECKPOINT_INDEX_PATH.with_suffix(
            ".json.tmp"
        )
    )

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            index,
            f,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temp_path,
        CHECKPOINT_INDEX_PATH,
    )


CHECKPOINT_INDEX = (
    load_checkpoint_index()
)

# =========================================================================
# SAVE LATEST
# =========================================================================
def save_latest_checkpoint(
    checkpoint,
):

    atomic_torch_save(
        checkpoint,
        LATEST_CHECKPOINT_PATH,
    )

    CHECKPOINT_INDEX[
        "latest_checkpoint"
    ] = str(
        LATEST_CHECKPOINT_PATH.relative_to(
            NOTEBOOK_DIR
        )
    )

    save_checkpoint_index(
        CHECKPOINT_INDEX
    )


# =========================================================================
# SAVE BEST
# =========================================================================
def save_best_checkpoint(
    checkpoint,
    step,
    validation_loss,
):

    # -------------------------------------------------------------
    # Always keep the single best model here.
    # -------------------------------------------------------------
    atomic_torch_save(
        checkpoint,
        BEST_CHECKPOINT_PATH,
    )

    # -------------------------------------------------------------
    # Preserve a historical best checkpoint.
    # -------------------------------------------------------------
    historical_path = (
        CHECKPOINTS_DIR
        / (
            f"best_step_{int(step):07d}"
            f"_loss_{float(validation_loss):.6f}.pt"
        )
    )

    atomic_torch_save(
        checkpoint,
        historical_path,
    )

    relative_path = str(
        historical_path.relative_to(
            NOTEBOOK_DIR
        )
    )

    CHECKPOINT_INDEX[
        "best_checkpoints"
    ].append(
        {
            "path": relative_path,
            "step": int(step),
            "validation_loss": float(
                validation_loss
            ),
        }
    )

    # -------------------------------------------------------------
    # Keep only the configured number of historical bests.
    # -------------------------------------------------------------
    while len(
        CHECKPOINT_INDEX[
            "best_checkpoints"
        ]
    ) > MAX_BEST_CHECKPOINTS:

        oldest = (
            CHECKPOINT_INDEX[
                "best_checkpoints"
            ].pop(0)
        )

        oldest_path = (
            NOTEBOOK_DIR
            / oldest["path"]
        )

        if (
            oldest_path.is_file()
            and oldest_path.resolve()
            != BEST_CHECKPOINT_PATH.resolve()
        ):

            oldest_path.unlink()

    save_checkpoint_index(
        CHECKPOINT_INDEX
    )


# =========================================================================
# LOAD LATEST COMPATIBLE CHECKPOINT
# =========================================================================
def load_latest_checkpoint():

    if not LATEST_CHECKPOINT_PATH.is_file():
        return None

    try:

        checkpoint = torch.load(
            LATEST_CHECKPOINT_PATH,
            map_location="cpu",
            weights_only=False,
        )

    except Exception as exc:

        print(
            "⚠️ Failed to load latest checkpoint."
        )

        print(
            f"   Reason: {exc}"
        )

        return None

    if not checkpoint_is_compatible(
        checkpoint
    ):

        print(
            "⚠️ Latest checkpoint is incompatible "
            "with the current Architecture-A configuration."
        )

        return None

    # -------------------------------------------------------------
    # Restore all training state.
    # -------------------------------------------------------------
    MODEL.load_state_dict(
        checkpoint["model"]
    )

    OPTIMIZER.load_state_dict(
        checkpoint["optimizer"]
    )

    LR_SCHEDULER.load_state_dict(
        checkpoint["scheduler"]
    )

    GRAD_SCALER.load_state_dict(
        checkpoint["scaler"]
    )

    TRAIN_SAMPLER.load_state_dict(
        checkpoint["train_sampler"]
    )

    VAL_SAMPLER.load_state_dict(
        checkpoint["val_sampler"]
    )

    restore_rng_state(
        checkpoint["rng_state"]
    )

    print(
        "♻️ Compatible checkpoint loaded."
    )

    print(
        f"   Step          : "
        f"{checkpoint['step']:,}"
    )

    print(
        f"   Tokens seen   : "
        f"{checkpoint['tokens_seen']:,}"
    )

    print(
        f"   Best val loss : "
        f"{checkpoint['best_validation_loss']:.6f}"
    )

    print(
        f"   Current LR    : "
        f"{checkpoint['current_lr']:.6g}"
    )

    return checkpoint


# =========================================================================
# INSPECT EXISTING CHECKPOINTS
# =========================================================================
def inspect_existing_checkpoints():

    compatible = []
    incompatible = []

    for checkpoint_path in sorted(
        CHECKPOINTS_DIR.glob("*.pt")
    ):

        try:

            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=False,
            )

            relative_path = str(
                checkpoint_path.relative_to(
                    NOTEBOOK_DIR
                )
            )

            if checkpoint_is_compatible(
                checkpoint
            ):

                compatible.append(
                    relative_path
                )

            else:

                incompatible.append(
                    relative_path
                )

        except Exception:

            incompatible.append(
                str(
                    checkpoint_path.relative_to(
                        NOTEBOOK_DIR
                    )
                )
            )

    return {
        "compatible": compatible,
        "incompatible": incompatible,
    }


CHECKPOINT_INSPECTION = (
    inspect_existing_checkpoints()
)

save_checkpoint_index(
    CHECKPOINT_INDEX
)

# =========================================================================
# SUMMARY
# =========================================================================
print()
print(
    "✅ Architecture-A checkpoint infrastructure ready."
)

print()

print(
    f"Run mode                 : "
    f"{RUN_MODE}"
)

print(
    f"Schema version            : "
    f"{CHECKPOINT_SCHEMA_VERSION}"
)

print(
    f"Latest checkpoint         : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Best checkpoint           : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Max historical bests     : "
    f"{MAX_BEST_CHECKPOINTS}"
)

print(
    f"Compatible checkpoints   : "
    f"{len(CHECKPOINT_INSPECTION['compatible'])}"
)

print(
    f"Incompatible checkpoints : "
    f"{len(CHECKPOINT_INSPECTION['incompatible'])}"
)

print()

print(
    "Resume policy             : "
    "exact configuration match only"
)

print(
    "Saved state               : "
    "model + optimizer + scheduler + scaler"
)

print(
    "                            + samplers + RNG"
)

print(
    "                            + progress + history"
)

💾 Building Architecture-A checkpoint infrastructure...

✅ Architecture-A checkpoint infrastructure ready.

Run mode                 : production
Schema version            : 2
Latest checkpoint         : /kaggle/working/architecture_A/checkpoints/latest.pt
Best checkpoint           : /kaggle/working/architecture_A/checkpoints/best.pt
Max historical bests     : 12
Compatible checkpoints   : 0
Incompatible checkpoints : 0

Resume policy             : exact configuration match only
Saved state               : model + optimizer + scheduler + scaler
                            + samplers + RNG
                            + progress + history


In [8]:
# Cell 07.5 — Restore Architecture-A checkpoint from Phase11
# DECISION: ADD

from pathlib import Path
import shutil
import torch

print("📦 Restoring Architecture-A checkpoint from Phase11...")
print()

INPUT_ROOT = Path("/kaggle/input")
TARGET_ROOT = Path("/kaggle/working/architecture_A")
TARGET_CHECKPOINT_DIR = TARGET_ROOT / "checkpoints"

TARGET_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# =========================================================================
# FIND ATTACHED DATASET CHECKPOINT
# =========================================================================

latest_candidates = sorted(
    [
        p
        for p in INPUT_ROOT.rglob("latest.pt")
        if "architecture_A" in p.parts
        and "checkpoints" in p.parts
    ]
)

if not latest_candidates:

    raise FileNotFoundError(
        "❌ Could not find Architecture-A latest.pt "
        "inside /kaggle/input.\n"
        "Make sure Phase11 is attached to this notebook."
    )

print("✅ Found checkpoint candidate(s):")

for path in latest_candidates:

    print(
        f"   {path}"
    )

# =========================================================================
# SELECT SOURCE CHECKPOINT
# =========================================================================

SOURCE_LATEST = (
    latest_candidates[0]
)

SOURCE_CHECKPOINT_DIR = (
    SOURCE_LATEST.parent
)

print()
print(
    "Using checkpoint directory:"
)

print(
    f"   {SOURCE_CHECKPOINT_DIR}"
)

# =========================================================================
# VERIFY CHECKPOINT BEFORE COPYING
# =========================================================================

print()
print(
    "🔍 Inspecting checkpoint..."
)

checkpoint = torch.load(
    SOURCE_LATEST,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(
    checkpoint,
    dict,
), "❌ Checkpoint is not a dictionary."

print(
    f"   Schema version : "
    f"{checkpoint.get('checkpoint_schema_version')}"
)

print(
    f"   Architecture   : "
    f"{checkpoint.get('architecture_name')}"
)

print(
    f"   Run mode       : "
    f"{checkpoint.get('run_mode')}"
)

print(
    f"   Step           : "
    f"{checkpoint.get('step', 'MISSING'):,}"
)

print(
    f"   Tokens seen    : "
    f"{checkpoint.get('tokens_seen', 'MISSING'):,}"
)

print(
    f"   Best val loss  : "
    f"{checkpoint.get('best_validation_loss', float('inf')):.6f}"
)

# =========================================================================
# COMPATIBILITY CHECK
# =========================================================================

assert checkpoint.get(
    "architecture_name"
) == ARCHITECTURE_NAME, (
    "❌ Checkpoint architecture does not match "
    "the current Architecture-A configuration."
)

assert checkpoint.get(
    "architecture_version"
) == ARCHITECTURE_VERSION, (
    "❌ Architecture version mismatch."
)

assert checkpoint.get(
    "experiment_id"
) == EXPERIMENT_ID, (
    "❌ Experiment ID mismatch."
)

assert checkpoint.get(
    "run_mode"
) == RUN_MODE, (
    "❌ RUN_MODE mismatch. "
    "Set RUN_MODE='production'."
)

assert checkpoint.get(
    "config_fingerprint"
) == CONFIG_FINGERPRINT, (
    "❌ Configuration fingerprint mismatch.\n"
    "Do not change the Architecture-A production configuration."
)

assert (
    "model" in checkpoint
    and "optimizer" in checkpoint
    and "scheduler" in checkpoint
    and "scaler" in checkpoint
    and "train_sampler" in checkpoint
    and "val_sampler" in checkpoint
    and "rng_state" in checkpoint
), (
    "❌ Checkpoint is missing required training state."
)

print()
print(
    "✅ Checkpoint is fully compatible."
)

del checkpoint

# =========================================================================
# COPY LATEST CHECKPOINT
# =========================================================================

TARGET_LATEST = (
    TARGET_CHECKPOINT_DIR
    / "latest.pt"
)

print()
print(
    "⏳ Copying latest.pt into /kaggle/working..."
)

shutil.copy2(
    SOURCE_LATEST,
    TARGET_LATEST,
)

print(
    "✅ latest.pt restored."
)

# =========================================================================
# COPY BEST CHECKPOINT
# =========================================================================

SOURCE_BEST = (
    SOURCE_CHECKPOINT_DIR
    / "best.pt"
)

TARGET_BEST = (
    TARGET_CHECKPOINT_DIR
    / "best.pt"
)

if SOURCE_BEST.exists():

    print(
        "⏳ Copying best.pt..."
    )

    shutil.copy2(
        SOURCE_BEST,
        TARGET_BEST,
    )

    print(
        "✅ best.pt restored."
    )

else:

    print(
        "⚠️ best.pt not found in Phase11."
    )

# =========================================================================
# FINAL VERIFICATION
# =========================================================================

print()
print("=" * 72)
print("✅ ARCHITECTURE-A CHECKPOINT RESTORE COMPLETE")
print("=" * 72)

print(
    f"latest.pt : "
    f"{TARGET_LATEST.exists()}"
)

if TARGET_LATEST.exists():

    print(
        f"size      : "
        f"{TARGET_LATEST.stat().st_size / (1024**3):.2f} GB"
    )

print(
    f"best.pt   : "
    f"{TARGET_BEST.exists()}"
)

print(
    f"path      : "
    f"{TARGET_LATEST}"
)

print("=" * 72)

📦 Restoring Architecture-A checkpoint from Phase11...

✅ Found checkpoint candidate(s):
   /kaggle/input/datasets/rohitbatra2004/phase111/architecture_A/checkpoints/latest.pt

Using checkpoint directory:
   /kaggle/input/datasets/rohitbatra2004/phase111/architecture_A/checkpoints

🔍 Inspecting checkpoint...
   Schema version : 2
   Architecture   : Architecture-A-Decoder-Only-Transformer
   Run mode       : production
   Step           : 6,104
   Tokens seen    : 100,007,936
   Best val loss  : 3.010440

✅ Checkpoint is fully compatible.

⏳ Copying latest.pt into /kaggle/working...
✅ latest.pt restored.
⏳ Copying best.pt...
✅ best.pt restored.

✅ ARCHITECTURE-A CHECKPOINT RESTORE COMPLETE
latest.pt : True
size      : 1.40 GB
best.pt   : True
path      : /kaggle/working/architecture_A/checkpoints/latest.pt


In [9]:
# Cell 08 — Define Architecture-A validation and resumable training engine
# DECISION: ADD

import time
import math
from contextlib import nullcontext

import torch

print("🧠 Defining Architecture-A training engine...")


# =========================================================================
# VALIDATION
# =========================================================================
@torch.no_grad()
def evaluate_architecture_A(
    model,
    sampler,
    batches,
    batch_size,
):
    """
    Evaluate mean next-token cross-entropy over a fixed number of
    sampled validation batches.

    Validation sampling uses its own checkpointable RNG state.
    """

    model.eval()

    total_loss = 0.0
    total_batches = 0

    for _ in range(
        int(batches)
    ):

        input_ids, targets, _ = (
            sampler.sample_batch(
                batch_size=int(
                    batch_size
                ),
                device=DEVICE,
            )
        )

        with autocast_context():

            _, loss = model(
                input_ids,
                targets=targets,
            )

        assert loss is not None

        loss_value = float(
            loss.detach()
            .float()
            .cpu()
            .item()
        )

        assert math.isfinite(
            loss_value
        ), (
            "Validation loss became non-finite."
        )

        total_loss += loss_value
        total_batches += 1

    assert total_batches > 0

    mean_loss = (
        total_loss
        / total_batches
    )

    perplexity = math.exp(
        min(
            mean_loss,
            20.0,
        )
    )

    return {
        "loss": float(
            mean_loss
        ),
        "perplexity": float(
            perplexity
        ),
        "batches": int(
            total_batches
        ),
    }


# =========================================================================
# TRAINING STATE INITIALIZATION
# =========================================================================
def create_fresh_training_state():

    return {
        "step": 0,

        "tokens_seen": 0,

        "train_loss": float("nan"),

        "best_validation_loss": float("inf"),

        "best_validation_perplexity": float("inf"),

        "best_validation_step": 0,

        "bad_validation_count": 0,

        "best_checkpoint_count": 0,

        "status": "initialized",

        "history": [],
    }


# =========================================================================
# RESUME / INITIALIZE STATE
# =========================================================================
def initialize_training_state():

    checkpoint = (
        load_latest_checkpoint()
    )

    if checkpoint is None:

        state = (
            create_fresh_training_state()
        )

        print()
        print(
            "🆕 Starting Architecture-A "
            "from a fresh model state."
        )

        return state

    # ---------------------------------------------------------------------
    # Extract all persistent training state.
    # ---------------------------------------------------------------------
    history = checkpoint.get(
        "history",
        [],
    )

    if not isinstance(
        history,
        list,
    ):
        history = []

    state = {
        "step": int(
            checkpoint.get(
                "step",
                0,
            )
        ),

        "tokens_seen": int(
            checkpoint.get(
                "tokens_seen",
                0,
            )
        ),

        "train_loss": float(
            checkpoint.get(
                "train_loss",
                float("nan"),
            )
        ),

        "best_validation_loss": float(
            checkpoint.get(
                "best_validation_loss",
                float("inf"),
            )
        ),

        "best_validation_perplexity": (
            float("inf")
        ),

        "best_validation_step": int(
            checkpoint.get(
                "best_validation_step",
                0,
            )
        ),

        "bad_validation_count": int(
            checkpoint.get(
                "bad_validation_count",
                0,
            )
        ),

        "best_checkpoint_count": int(
            checkpoint.get(
                "best_checkpoint_count",
                0,
            )
        ),

        "status": str(
            checkpoint.get(
                "status",
                "resumed",
            )
        ),

        "history": history,
    }

    # ---------------------------------------------------------------------
    # Recover best validation PPL from history whenever available.
    # ---------------------------------------------------------------------
    validation_perplexities = [
        float(
            item["validation_perplexity"]
        )
        for item in history
        if (
            isinstance(item, dict)
            and
            "validation_perplexity" in item
        )
        and
        math.isfinite(
            float(
                item["validation_perplexity"]
            )
        )
    ]

    if validation_perplexities:

        state[
            "best_validation_perplexity"
        ] = min(
            validation_perplexities
        )

    print()
    print(
        "♻️ Training state resumed."
    )

    print(
        f"   Step                : "
        f"{state['step']:,}"
    )

    print(
        f"   Tokens seen         : "
        f"{state['tokens_seen']:,}"
    )

    print(
        f"   Best validation     : "
        f"{state['best_validation_loss']:.6f}"
    )

    print(
        f"   Best checkpoint cnt : "
        f"{state['best_checkpoint_count']}"
    )

    print(
        f"   Bad validations     : "
        f"{state['bad_validation_count']}"
    )

    return state


# =========================================================================
# UPDATE LEARNING RATE SAFELY
# =========================================================================
def current_learning_rate():

    return float(
        OPTIMIZER.param_groups[
            0
        ]["lr"]
    )


# =========================================================================
# SAVE TRAINING STATE
# =========================================================================
def persist_training_state(
    state,
    status,
):

    state["status"] = str(
        status
    )

    checkpoint = build_checkpoint(
        step=state["step"],
        tokens_seen=state["tokens_seen"],
        train_loss=state["train_loss"],
        best_validation_loss=(
            state["best_validation_loss"]
        ),
        best_validation_step=(
            state["best_validation_step"]
        ),
        bad_validation_count=(
            state["bad_validation_count"]
        ),
        best_checkpoint_count=(
            state["best_checkpoint_count"]
        ),
        history=state["history"],
        status=status,
    )

    save_latest_checkpoint(
        checkpoint
    )

    return checkpoint


# =========================================================================
# DECIDE WHETHER EARLY STOPPING SHOULD ACTIVATE
# =========================================================================
def should_stop_early(
    state,
):

    if not EARLY_STOPPING_ENABLED:

        return False, ""

    # ---------------------------------------------------------------------
    # Never stop before the minimum number of best checkpoints.
    # ---------------------------------------------------------------------
    if (
        state[
            "best_checkpoint_count"
        ]
        < MIN_BEST_CHECKPOINTS_BEFORE_STOP
    ):

        return False, (
            "minimum-best-checkpoint requirement"
        )

    # ---------------------------------------------------------------------
    # Hard cap: even continuously decreasing validation loss cannot make
    # the experiment continue forever.
    # ---------------------------------------------------------------------
    if (
        state[
            "best_checkpoint_count"
        ]
        >= MAX_BEST_CHECKPOINTS
    ):

        return True, (
            "maximum-best-checkpoint limit reached"
        )

    # ---------------------------------------------------------------------
    # Plateau-based stopping.
    # ---------------------------------------------------------------------
    if (
        state[
            "step"
        ]
        >= MIN_STEPS_BEFORE_EARLY_STOP
    ):

        if (
            state[
                "bad_validation_count"
            ]
            >= EARLY_STOPPING_PATIENCE
        ):

            return True, (
                "validation plateau reached"
            )

    return False, ""


# =========================================================================
# TRAINING ENGINE
# =========================================================================
def run_architecture_A_training():

    print()
    print(
        "🚀 Starting Architecture-A training engine..."
    )

    # =====================================================================
    # RESTORE OR INITIALIZE
    # =====================================================================
    state = (
        initialize_training_state()
    )

    # ---------------------------------------------------------------------
    # If a previous run already finished, do not retrain it.
    # ---------------------------------------------------------------------
    if state["status"] in {
        "early_stopped",
        "max_steps_reached",
        "completed",
    }:

        print()
        print(
            "ℹ️ Compatible completed training state "
            "already exists."
        )

        print(
            f"   Status: {state['status']}"
        )

        return state

    # =====================================================================
    # TRAINING LOOP
    # =====================================================================
    MODEL.train()

    training_start = time.perf_counter()

    last_step_time = time.perf_counter()

    try:

        while (
            state["step"]
            < MAX_STEPS
        ):

            # -------------------------------------------------------------
            # Stop check at loop boundary.
            # -------------------------------------------------------------
            should_stop, reason = (
                should_stop_early(
                    state
                )
            )

            if should_stop:

                print()
                print(
                    "🛑 Early stopping activated."
                )

                print(
                    f"   Reason: {reason}"
                )

                persist_training_state(
                    state,
                    "early_stopped",
                )

                break

            # -------------------------------------------------------------
            # Start one optimizer update.
            # -------------------------------------------------------------
            OPTIMIZER.zero_grad(
                set_to_none=True
            )

            accumulated_loss = 0.0

            # -------------------------------------------------------------
            # Gradient accumulation.
            # -------------------------------------------------------------
            for _micro_step in range(
                GRAD_ACCUM_STEPS
            ):

                input_ids, targets, _ = (
                    TRAIN_SAMPLER.sample_batch(
                        batch_size=(
                            MICRO_BATCH_SIZE
                        ),
                        device=DEVICE,
                    )
                )

                with autocast_context():

                    _, loss = MODEL(
                        input_ids,
                        targets=targets,
                    )

                    # Normalize by accumulation steps.
                    scaled_loss = (
                        loss
                        / GRAD_ACCUM_STEPS
                    )

                assert loss is not None

                loss_value = float(
                    loss.detach()
                    .float()
                    .cpu()
                    .item()
                )

                assert math.isfinite(
                    loss_value
                ), (
                    "Training loss became "
                    "non-finite."
                )

                accumulated_loss += (
                    loss_value
                )

                # ---------------------------------------------------------
                # Backward.
                # ---------------------------------------------------------
                if AMP_MODE == "fp16":

                    GRAD_SCALER.scale(
                        scaled_loss
                    ).backward()

                else:

                    scaled_loss.backward()

            # -------------------------------------------------------------
            # Gradient clipping + optimizer update.
            # -------------------------------------------------------------
            if AMP_MODE == "fp16":

                GRAD_SCALER.unscale_(
                    OPTIMIZER
                )

            grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    MODEL.parameters(),
                    GRAD_CLIP_NORM,
                )
            )

            if AMP_MODE == "fp16":

                GRAD_SCALER.step(
                    OPTIMIZER
                )

                GRAD_SCALER.update()

            else:

                OPTIMIZER.step()

            # -------------------------------------------------------------
            # Advance scheduler AFTER optimizer step.
            # -------------------------------------------------------------
            LR_SCHEDULER.step()

            # -------------------------------------------------------------
            # Update counters.
            # -------------------------------------------------------------
            state["step"] += 1

            state["tokens_seen"] += (
                TOKENS_PER_OPTIMIZER_STEP
            )

            state["train_loss"] = (
                accumulated_loss
                / GRAD_ACCUM_STEPS
            )

            now = time.perf_counter()

            step_seconds = (
                now
                - last_step_time
            )

            last_step_time = now

            # -------------------------------------------------------------
            # Logging.
            # -------------------------------------------------------------
            if (
                state["step"]
                % LOG_EVERY_STEPS
                == 0
            ):

                print(
                    f"[step "
                    f"{state['step']:>6}/"
                    f"{MAX_STEPS:<6}] "
                    f"train_loss="
                    f"{state['train_loss']:.5f} | "
                    f"lr="
                    f"{current_learning_rate():.6g} | "
                    f"grad_norm="
                    f"{float(grad_norm):.4f} | "
                    f"time="
                    f"{step_seconds:.2f}s"
                )

            # =============================================================
            # VALIDATION
            # =============================================================
            if (
                state["step"]
                % EVAL_EVERY_STEPS
                == 0
            ):

                validation = (
                    evaluate_architecture_A(
                        model=MODEL,
                        sampler=VAL_SAMPLER,
                        batches=VAL_EVAL_BATCHES,
                        batch_size=VAL_BATCH_SIZE,
                    )
                )

                validation_loss = float(
                    validation["loss"]
                )

                validation_ppl = float(
                    validation["perplexity"]
                )

                previous_best = float(
                    state[
                        "best_validation_loss"
                    ]
                )

                meaningful_improvement = (
                    not math.isfinite(
                        previous_best
                    )
                    or
                    validation_loss
                    < (
                        previous_best
                        - EARLY_STOPPING_MIN_DELTA
                    )
                )

                # ---------------------------------------------------------
                # Record validation result.
                # ---------------------------------------------------------
                validation_record = {
                    "step": int(
                        state["step"]
                    ),

                    "tokens_seen": int(
                        state["tokens_seen"]
                    ),

                    "train_loss": float(
                        state["train_loss"]
                    ),

                    "validation_loss": (
                        validation_loss
                    ),

                    "validation_perplexity": (
                        validation_ppl
                    ),

                    "learning_rate": (
                        current_learning_rate()
                    ),

                    "seconds_per_step": (
                        float(step_seconds)
                    ),

                    "meaningful_improvement": (
                        bool(
                            meaningful_improvement
                        )
                    ),
                }

                state[
                    "history"
                ].append(
                    validation_record
                )

                # ---------------------------------------------------------
                # BEST CHECKPOINT
                # ---------------------------------------------------------
                if meaningful_improvement:

                    state[
                        "best_validation_loss"
                    ] = validation_loss

                    state[
                        "best_validation_perplexity"
                    ] = validation_ppl

                    state[
                        "best_validation_step"
                    ] = state["step"]

                    state[
                        "bad_validation_count"
                    ] = 0

                    state[
                        "best_checkpoint_count"
                    ] += 1

                    best_checkpoint = (
                        build_checkpoint(
                            step=state["step"],
                            tokens_seen=(
                                state["tokens_seen"]
                            ),
                            train_loss=(
                                state["train_loss"]
                            ),
                            best_validation_loss=(
                                state[
                                    "best_validation_loss"
                                ]
                            ),
                            best_validation_step=(
                                state[
                                    "best_validation_step"
                                ]
                            ),
                            bad_validation_count=(
                                state[
                                    "bad_validation_count"
                                ]
                            ),
                            best_checkpoint_count=(
                                state[
                                    "best_checkpoint_count"
                                ]
                            ),
                            history=(
                                state["history"]
                            ),
                            status="best",
                        )
                    )

                    save_best_checkpoint(
                        best_checkpoint,
                        state["step"],
                        validation_loss,
                    )

                    print()
                    print(
                        "⭐ NEW BEST"
                    )

                    print(
                        f"   val_loss : "
                        f"{validation_loss:.6f}"
                    )

                    print(
                        f"   val_ppl  : "
                        f"{validation_ppl:.4f}"
                    )

                    print(
                        f"   best count: "
                        f"{state['best_checkpoint_count']}/"
                        f"{MAX_BEST_CHECKPOINTS}"
                    )

                else:

                    state[
                        "bad_validation_count"
                    ] += 1

                # ---------------------------------------------------------
                # ALWAYS SAVE LATEST AT EVAL CHECKPOINT.
                # ---------------------------------------------------------
                persist_training_state(
                    state,
                    "running",
                )

                # ---------------------------------------------------------
                # Validation report.
                # ---------------------------------------------------------
                print()
                print(
                    "📊 Validation"
                )

                print(
                    f"   Step        : "
                    f"{state['step']:,}"
                )

                print(
                    f"   Train loss  : "
                    f"{state['train_loss']:.6f}"
                )

                print(
                    f"   Val loss    : "
                    f"{validation_loss:.6f}"
                )

                print(
                    f"   Val PPL     : "
                    f"{validation_ppl:.4f}"
                )

                print(
                    f"   Best count  : "
                    f"{state['best_checkpoint_count']}/"
                    f"{MAX_BEST_CHECKPOINTS}"
                )

                print(
                    f"   Bad evals   : "
                    f"{state['bad_validation_count']}/"
                    f"{EARLY_STOPPING_PATIENCE}"
                )

                # ---------------------------------------------------------
                # Decide whether to stop AFTER saving the latest state.
                # ---------------------------------------------------------
                should_stop, reason = (
                    should_stop_early(
                        state
                    )
                )

                if should_stop:

                    print()
                    print(
                        "🛑 Early stopping activated."
                    )

                    print(
                        f"   Reason: {reason}"
                    )

                    persist_training_state(
                        state,
                        "early_stopped",
                    )

                    state[
                        "status"
                    ] = "early_stopped"

                    break

            # =============================================================
            # NON-EVALUATION CHECKPOINT
            # =============================================================
            elif (
                state["step"]
                % CHECKPOINT_EVERY_STEPS
                == 0
            ):

                persist_training_state(
                    state,
                    "running",
                )

        # =================================================================
        # MAX-STEPS TERMINATION
        # =================================================================
        if (
            state["step"]
            >= MAX_STEPS
            and
            state["status"]
            not in {
                "early_stopped",
                "completed",
            }
        ):

            state["status"] = (
                "max_steps_reached"
            )

            persist_training_state(
                state,
                "max_steps_reached",
            )

            print()
            print(
                "🏁 Maximum training-step ceiling reached."
            )

    except KeyboardInterrupt:

        # -----------------------------------------------------------------
        # Preserve whatever progress exists before leaving the cell.
        # -----------------------------------------------------------------
        print()
        print(
            "⏹️ Training interrupted by user."
        )

        persist_training_state(
            state,
            "interrupted",
        )

        raise

    except Exception:

        # -----------------------------------------------------------------
        # Preserve a recoverable checkpoint if possible.
        # -----------------------------------------------------------------
        try:

            persist_training_state(
                state,
                "error_recoverable",
            )

        except Exception:

            pass

        raise

    # =========================================================================
    # FINAL SUMMARY
    # =========================================================================
    elapsed_seconds = (
        time.perf_counter()
        - training_start
    )

    if state["step"] > 0:

        average_seconds_per_step = (
            elapsed_seconds
            / state["step"]
        )

    else:

        average_seconds_per_step = 0.0

    state[
        "elapsed_seconds"
    ] = float(
        elapsed_seconds
    )

    state[
        "average_seconds_per_step"
    ] = float(
        average_seconds_per_step
    )

    print()
    print(
        "✅ Architecture-A training engine finished."
    )

    print()

    print(
        f"Run mode              : "
        f"{RUN_MODE}"
    )

    print(
        f"Status                : "
        f"{state['status']}"
    )

    print(
        f"Optimizer steps       : "
        f"{state['step']:,}"
    )

    print(
        f"Tokens seen           : "
        f"{state['tokens_seen']:,}"
    )

    print(
        f"Best validation loss  : "
        f"{state['best_validation_loss']:.6f}"
    )

    print(
        f"Best validation PPL   : "
        f"{state['best_validation_perplexity']:.4f}"
    )

    print(
        f"Best validation step  : "
        f"{state['best_validation_step']:,}"
    )

    print(
        f"Best checkpoints      : "
        f"{state['best_checkpoint_count']}"
    )

    print(
        f"Elapsed time (sec)    : "
        f"{elapsed_seconds:.2f}"
    )

    print(
        f"Average sec/step      : "
        f"{average_seconds_per_step:.2f}"
    )

    return state


# =========================================================================
# ENGINE READY
# =========================================================================
print()
print(
    "✅ Architecture-A training engine defined."
)

print(
    f"Mode                    : {RUN_MODE}"
)

print(
    f"Maximum steps            : {MAX_STEPS:,}"
)

print(
    f"Validation every         : "
    f"{EVAL_EVERY_STEPS:,} steps"
)

print(
    f"Early stopping           : "
    f"{EARLY_STOPPING_ENABLED}"
)

print(
    f"Minimum best checkpoints : "
    f"{MIN_BEST_CHECKPOINTS_BEFORE_STOP}"
)

print(
    f"Maximum best checkpoints : "
    f"{MAX_BEST_CHECKPOINTS}"
)

print(
    f"Patience                 : "
    f"{EARLY_STOPPING_PATIENCE}"
)

print(
    f"Minimum meaningful Δloss : "
    f"{EARLY_STOPPING_MIN_DELTA}"
)

🧠 Defining Architecture-A training engine...

✅ Architecture-A training engine defined.
Mode                    : production
Maximum steps            : 6,104
Validation every         : 500 steps
Early stopping           : True
Minimum best checkpoints : 8
Maximum best checkpoints : 12
Patience                 : 4
Minimum meaningful Δloss : 0.005


In [10]:
# Cell 09 — Execute Architecture-A training in the selected mode
# DECISION: ADD

import json
from pathlib import Path

print("🚀 Launching Architecture-A training...")
print()

# =========================================================================
# RUN TRAINING
# =========================================================================
TRAINING_STATE = run_architecture_A_training()

# =========================================================================
# TRAINING SUMMARY
# =========================================================================
TRAINING_SUMMARY = {
    "project": "CS781 Phase 1",

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": (
        RUN_MODE
    ),

    "model_type": (
        "decoder-only Transformer"
    ),

    "status": (
        TRAINING_STATE["status"]
    ),

    "optimizer_steps": int(
        TRAINING_STATE["step"]
    ),

    "tokens_seen": int(
        TRAINING_STATE["tokens_seen"]
    ),

    "max_steps": int(
        MAX_STEPS
    ),

    "train_tokens_ceiling": int(
        TRAIN_TOKENS
    ),

    "validation_tokens": int(
        VAL_TOKENS
    ),

    "train_loss": float(
        TRAINING_STATE["train_loss"]
    ),

    "best_validation_loss": float(
        TRAINING_STATE[
            "best_validation_loss"
        ]
    ),

    "best_validation_perplexity": float(
        TRAINING_STATE[
            "best_validation_perplexity"
        ]
    ),

    "best_validation_step": int(
        TRAINING_STATE[
            "best_validation_step"
        ]
    ),

    "best_checkpoint_count": int(
        TRAINING_STATE[
            "best_checkpoint_count"
        ]
    ),

    "bad_validation_count": int(
        TRAINING_STATE[
            "bad_validation_count"
        ]
    ),

    "early_stopping": {
        "enabled": bool(
            EARLY_STOPPING_ENABLED
        ),

        "minimum_best_checkpoints": int(
            MIN_BEST_CHECKPOINTS_BEFORE_STOP
        ),

        "maximum_best_checkpoints": int(
            MAX_BEST_CHECKPOINTS
        ),

        "patience": int(
            EARLY_STOPPING_PATIENCE
        ),

        "min_delta": float(
            EARLY_STOPPING_MIN_DELTA
        ),

        "minimum_steps": int(
            MIN_STEPS_BEFORE_EARLY_STOP
        ),
    },

    "elapsed_seconds": float(
        TRAINING_STATE.get(
            "elapsed_seconds",
            0.0,
        )
    ),

    "average_seconds_per_step": float(
        TRAINING_STATE.get(
            "average_seconds_per_step",
            0.0,
        )
    ),

    "history": (
        TRAINING_STATE["history"]
    ),
}

# =========================================================================
# SAVE TRAINING SUMMARY
# =========================================================================
TRAINING_SUMMARY_PATH = (
    METRICS_DIR /
    "training_summary_A.json"
)

with open(
    TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_SUMMARY,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =========================================================================
# FINAL OUTPUT
# =========================================================================
print()
print(
    "✅ Architecture-A training run recorded."
)

print()

print(
    f"Run mode              : "
    f"{RUN_MODE}"
)

print(
    f"Status                : "
    f"{TRAINING_SUMMARY['status']}"
)

print(
    f"Optimizer steps       : "
    f"{TRAINING_SUMMARY['optimizer_steps']:,}"
)

print(
    f"Tokens seen           : "
    f"{TRAINING_SUMMARY['tokens_seen']:,}"
)

print(
    f"Best validation loss  : "
    f"{TRAINING_SUMMARY['best_validation_loss']:.6f}"
)

print(
    f"Best validation PPL   : "
    f"{TRAINING_SUMMARY['best_validation_perplexity']:.4f}"
)

print(
    f"Best checkpoint count : "
    f"{TRAINING_SUMMARY['best_checkpoint_count']}"
)

print(
    f"Training summary     : "
    f"{TRAINING_SUMMARY_PATH}"
)

🚀 Launching Architecture-A training...


🚀 Starting Architecture-A training engine...
♻️ Compatible checkpoint loaded.
   Step          : 6,104
   Tokens seen   : 100,007,936
   Best val loss : 3.010440
   Current LR    : 3e-05

♻️ Training state resumed.
   Step                : 6,104
   Tokens seen         : 100,007,936
   Best validation     : 3.010440
   Best checkpoint cnt : 8
   Bad validations     : 3

ℹ️ Compatible completed training state already exists.
   Status: max_steps_reached

✅ Architecture-A training run recorded.

Run mode              : production
Status                : max_steps_reached
Optimizer steps       : 6,104
Tokens seen           : 100,007,936
Best validation loss  : 3.010440
Best validation PPL   : 20.2963
Best checkpoint count : 8
Training summary     : /kaggle/working/architecture_A/metrics/training_summary_A.json


In [11]:
# Cell 10 — Select and freeze the best Architecture-A checkpoint
# DECISION: ADD

import json
import shutil
from pathlib import Path

import torch

print("🏆 Selecting best Architecture-A checkpoint...")

# =========================================================================
# REQUIRED TRAINING STATE
# =========================================================================
assert "TRAINING_STATE" in globals(), (
    "TRAINING_STATE is unavailable. "
    "Run Cell 09 first."
)

assert BEST_CHECKPOINT_PATH.is_file(), (
    "Best checkpoint does not exist."
)

assert CHECKPOINT_INDEX_PATH.is_file(), (
    "Checkpoint index does not exist."
)

# =========================================================================
# LOAD BEST CHECKPOINT
# =========================================================================
BEST_CHECKPOINT = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint_is_compatible(
    BEST_CHECKPOINT
), (
    "Best checkpoint is incompatible with "
    "the current Architecture-A configuration."
)

# =========================================================================
# RESTORE BEST MODEL
# =========================================================================
MODEL.load_state_dict(
    BEST_CHECKPOINT["model"]
)

MODEL = MODEL.to(
    DEVICE
)

MODEL.eval()

# =========================================================================
# EXTRACT BEST METRICS
# =========================================================================
BEST_STEP = int(
    BEST_CHECKPOINT[
        "best_validation_step"
    ]
)

BEST_VAL_LOSS = float(
    BEST_CHECKPOINT[
        "best_validation_loss"
    ]
)

BEST_VAL_PPL = float(
    TRAINING_STATE[
        "best_validation_perplexity"
    ]
)

BEST_TOKENS_SEEN = int(
    BEST_CHECKPOINT[
        "tokens_seen"
    ]
)

BEST_CHECKPOINT_COUNT = int(
    BEST_CHECKPOINT[
        "best_checkpoint_count"
    ]
)

# =========================================================================
# SAVE A DEDICATED FINAL MODEL COPY
# =========================================================================
FINAL_MODEL_PATH = (
    NOTEBOOK_DIR /
    "model_A_final.pt"
)

FINAL_STATE_DICT_PATH = (
    NOTEBOOK_DIR /
    "model_A_state_dict.pt"
)

MODEL_METADATA_PATH = (
    NOTEBOOK_DIR /
    "model_A_metadata.json"
)

SELECTED_CHECKPOINT_PATH = (
    METRICS_DIR /
    "selected_checkpoint_A.json"
)

# -------------------------------------------------------------------------
# Full checkpoint copy.
# -------------------------------------------------------------------------
torch.save(
    BEST_CHECKPOINT,
    FINAL_MODEL_PATH,
)

# -------------------------------------------------------------------------
# State-dict-only copy.
# -------------------------------------------------------------------------
torch.save(
    BEST_CHECKPOINT[
        "model"
    ],
    FINAL_STATE_DICT_PATH,
)

# =========================================================================
# MODEL METADATA
# =========================================================================
MODEL_METADATA = {
    "project": "CS781 Phase 1",

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": RUN_MODE,

    "model_type": (
        "decoder-only Transformer"
    ),

    "parameter_count": int(
        PARAMETER_COUNT
    ),

    "parameter_millions": float(
        PARAMETER_MILLIONS
    ),

    "model_config": dict(
        MODEL_CONFIG
    ),

    "best_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),

    "best_validation_step": (
        BEST_STEP
    ),

    "best_validation_loss": (
        BEST_VAL_LOSS
    ),

    "best_validation_perplexity": (
        BEST_VAL_PPL
    ),

    "tokens_seen_at_best": (
        BEST_TOKENS_SEEN
    ),

    "best_checkpoint_count": (
        BEST_CHECKPOINT_COUNT
    ),

    "early_stopping": {
        "enabled": bool(
            EARLY_STOPPING_ENABLED
        ),

        "minimum_best_checkpoints": int(
            MIN_BEST_CHECKPOINTS_BEFORE_STOP
        ),

        "maximum_best_checkpoints": int(
            MAX_BEST_CHECKPOINTS
        ),

        "patience": int(
            EARLY_STOPPING_PATIENCE
        ),

        "min_delta": float(
            EARLY_STOPPING_MIN_DELTA
        ),

        "minimum_steps": int(
            MIN_STEPS_BEFORE_EARLY_STOP
        ),
    },

    "source_probabilities": {
        _name: float(
            SOURCE_PROBABILITIES[
                _name
            ]
        )
        for _name in SOURCE_NAMES
    },

    "tokenizer_vocab_size": int(
        TOKENIZER_VOCAB_SIZE
    ),

    "config_fingerprint": (
        CONFIG_FINGERPRINT
    ),
}

with open(
    MODEL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        MODEL_METADATA,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =========================================================================
# SELECTED CHECKPOINT RECORD
# =========================================================================
SELECTED_CHECKPOINT_RECORD = {
    "project": "CS781 Phase 1",

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": RUN_MODE,

    "selected_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),

    "selected_step": int(
        BEST_STEP
    ),

    "best_validation_loss": (
        BEST_VAL_LOSS
    ),

    "best_validation_perplexity": (
        BEST_VAL_PPL
    ),

    "tokens_seen": int(
        BEST_TOKENS_SEEN
    ),

    "best_checkpoint_count": int(
        BEST_CHECKPOINT_COUNT
    ),

    "status": "selected",
}

with open(
    SELECTED_CHECKPOINT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SELECTED_CHECKPOINT_RECORD,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =========================================================================
# FINAL FILE CHECKS
# =========================================================================
assert FINAL_MODEL_PATH.is_file()
assert FINAL_STATE_DICT_PATH.is_file()
assert MODEL_METADATA_PATH.is_file()
assert SELECTED_CHECKPOINT_PATH.is_file()

# =========================================================================
# SUMMARY
# =========================================================================
print()
print(
    "✅ Best Architecture-A checkpoint selected."
)

print()
print(
    f"Run mode             : "
    f"{RUN_MODE}"
)

print(
    f"Best step            : "
    f"{BEST_STEP:,}"
)

print(
    f"Best validation loss : "
    f"{BEST_VAL_LOSS:.6f}"
)

print(
    f"Best validation PPL  : "
    f"{BEST_VAL_PPL:.4f}"
)

print(
    f"Tokens at best       : "
    f"{BEST_TOKENS_SEEN:,}"
)

print(
    f"Best checkpoint count: "
    f"{BEST_CHECKPOINT_COUNT}"
)

print()
print(
    f"Final model          : "
    f"{FINAL_MODEL_PATH}"
)

print(
    f"State dict           : "
    f"{FINAL_STATE_DICT_PATH}"
)

print(
    f"Model metadata       : "
    f"{MODEL_METADATA_PATH}"
)

print(
    f"Selection record     : "
    f"{SELECTED_CHECKPOINT_PATH}"
)

🏆 Selecting best Architecture-A checkpoint...

✅ Best Architecture-A checkpoint selected.

Run mode             : production
Best step            : 4,500
Best validation loss : 3.010440
Best validation PPL  : 20.2963
Tokens at best       : 73,728,000
Best checkpoint count: 8

Final model          : /kaggle/working/architecture_A/model_A_final.pt
State dict           : /kaggle/working/architecture_A/model_A_state_dict.pt
Model metadata       : /kaggle/working/architecture_A/model_A_metadata.json
Selection record     : /kaggle/working/architecture_A/metrics/selected_checkpoint_A.json


In [12]:
# Cell 11 — Configure robust Architecture-A generation
# DECISION: UPDATE

import json

print("🎛️ Configuring robust Architecture-A generation...")

if RUN_MODE == "fast":

    GENERATION_CONFIG = {
        "target_words": 8,
        "max_new_tokens": 512,

        # FAST model is intentionally under-trained.
        # Use diverse sampling to prevent token-collapse.
        "temperature": 1.0,
        "top_p": 1.0,
        "top_k": 100,
        "repetition_penalty": 1.20,

        "test_examples": 2,
    }

else:

    GENERATION_CONFIG = {
        "target_words": 128,
        "max_new_tokens": 1536,

        # Production uses the intended conservative decoding.
        "temperature": 0.10,
        "top_p": 0.95,
        "top_k": 50,
        "repetition_penalty": 1.0,

        "test_examples": 1000,
    }

# -------------------------------------------------------------------------
# Sanity checks
# -------------------------------------------------------------------------

assert GENERATION_CONFIG["target_words"] > 0
assert GENERATION_CONFIG["max_new_tokens"] > 0

assert 0.0 < GENERATION_CONFIG["temperature"] <= 10.0
assert 0.0 < GENERATION_CONFIG["top_p"] <= 1.0
assert GENERATION_CONFIG["top_k"] >= 0
assert GENERATION_CONFIG["repetition_penalty"] >= 1.0

if RUN_MODE == "production":
    assert GENERATION_CONFIG["target_words"] == 128
    assert GENERATION_CONFIG["test_examples"] == 1000

# -------------------------------------------------------------------------
# Save configuration
# -------------------------------------------------------------------------

GENERATION_CONFIG_MANIFEST = {
    "project": "CS781 Phase 1",
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "run_mode": RUN_MODE,
    "generation": dict(GENERATION_CONFIG),
    "official_requirement": {
        "prompt_words": 512,
        "output_words": 128,
        "prompt_excluded_from_output": True,
    },
}

GENERATION_CONFIG_PATH = (
    CONFIGS_DIR / "architecture_A_generation_config.json"
)

with open(
    GENERATION_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        GENERATION_CONFIG_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )

print()
print("✅ Generation configuration updated")
print(f"Run mode           : {RUN_MODE}")
print(f"Target words       : {GENERATION_CONFIG['target_words']}")
print(f"Maximum new tokens : {GENERATION_CONFIG['max_new_tokens']}")
print(f"Temperature        : {GENERATION_CONFIG['temperature']}")
print(f"Top-p              : {GENERATION_CONFIG['top_p']}")
print(f"Top-k              : {GENERATION_CONFIG['top_k']}")
print(
    f"Repetition penalty : "
    f"{GENERATION_CONFIG['repetition_penalty']}"
)
print(f"Test examples      : {GENERATION_CONFIG['test_examples']}")

🎛️ Configuring robust Architecture-A generation...

✅ Generation configuration updated
Run mode           : production
Target words       : 128
Maximum new tokens : 1536
Temperature        : 0.1
Top-p              : 0.95
Top-k              : 50
Repetition penalty : 1.0
Test examples      : 1000


In [13]:
# Cell 12 — Robust Architecture-A generation with exact word control
# DECISION: UPDATE

import math
import re
import torch
import torch.nn.functional as F


# =========================================================================
# SPECIAL TOKEN DISCOVERY
# =========================================================================

def _get_special_token_ids():
    special_ids = set()

    for token_name in ["<unk>", "<bos>", "<eos>", "<pad>"]:
        try:
            token_id = TOKENIZER.token_to_id(token_name)
            if token_id is not None:
                special_ids.add(int(token_id))
        except Exception:
            pass

    return special_ids


SPECIAL_TOKEN_IDS_A = _get_special_token_ids()


# =========================================================================
# WORD COUNT
# =========================================================================

def _count_words(text):
    """
    Official evaluation uses whitespace-separated words.
    Keep the same convention for generation validation.
    """
    return len(text.strip().split())


# =========================================================================
# TOKEN SAMPLING
# =========================================================================

def _sample_next_token(
    logits,
    generated_ids,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):
    """
    Sample one token using temperature + top-k + top-p,
    with optional repetition penalty.
    """

    logits = logits.float().clone()

    # -------------------------------------------------------------
    # Repetition penalty
    # -------------------------------------------------------------
    if repetition_penalty > 1.0 and generated_ids:

        unique_ids = set(generated_ids)

        for token_id in unique_ids:

            if token_id < 0 or token_id >= logits.numel():
                continue

            if logits[token_id] < 0:
                logits[token_id] *= repetition_penalty
            else:
                logits[token_id] /= repetition_penalty

    # -------------------------------------------------------------
    # Prevent special tokens
    # -------------------------------------------------------------
    for token_id in SPECIAL_TOKEN_IDS_A:

        if 0 <= token_id < logits.numel():
            logits[token_id] = -float("inf")

    # -------------------------------------------------------------
    # Temperature
    # -------------------------------------------------------------
    temperature = max(float(temperature), 1e-5)

    logits = logits / temperature

    # -------------------------------------------------------------
    # Top-k
    # -------------------------------------------------------------
    if top_k > 0 and top_k < logits.numel():

        values, indices = torch.topk(
            logits,
            k=top_k,
        )

        filtered = torch.full_like(
            logits,
            -float("inf"),
        )

        filtered.scatter_(
            0,
            indices,
            values,
        )

        logits = filtered

    # -------------------------------------------------------------
    # Top-p
    # -------------------------------------------------------------
    if top_p < 1.0:

        sorted_logits, sorted_indices = torch.sort(
            logits,
            descending=True,
        )

        sorted_probs = F.softmax(
            sorted_logits,
            dim=-1,
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1,
        )

        remove_mask = cumulative_probs > top_p

        # Always retain at least the most probable token.
        remove_mask[0] = False

        sorted_logits[remove_mask] = -float("inf")

        filtered = torch.full_like(
            logits,
            -float("inf"),
        )

        filtered.scatter_(
            0,
            sorted_indices,
            sorted_logits,
        )

        logits = filtered

    # -------------------------------------------------------------
    # Final sampling
    # -------------------------------------------------------------
    probs = F.softmax(
        logits,
        dim=-1,
    )

    if (
        not torch.isfinite(probs).all()
        or probs.sum() <= 0
    ):
        # Extremely defensive fallback.
        next_token = torch.argmax(logits)
    else:
        next_token = torch.multinomial(
            probs,
            num_samples=1,
        ).item()

    return int(next_token)


# =========================================================================
# WORD-BOUNDARY RESCUE
# =========================================================================

def _find_word_advancing_token(
    logits,
    generated_ids,
    current_word_count,
    top_k_candidates=100,
):
    """
    FAST-model safeguard.

    When generation gets stuck producing tokens that do not create
    whitespace-separated words, inspect the strongest candidate tokens
    and choose one that actually advances the word count.

    This does NOT invent text; the selected token still comes directly
    from the model's vocabulary/logits.
    """

    logits = logits.float().clone()

    # Never select special tokens.
    for token_id in SPECIAL_TOKEN_IDS_A:

        if 0 <= token_id < logits.numel():
            logits[token_id] = -float("inf")

    candidate_k = min(
        top_k_candidates,
        logits.numel(),
    )

    candidate_values, candidate_indices = torch.topk(
        logits,
        k=candidate_k,
    )

    best_token = None
    best_score = -float("inf")

    # Decode only candidate continuations.
    for score, token_tensor in zip(
        candidate_values.tolist(),
        candidate_indices.tolist(),
    ):

        token_id = int(token_tensor)

        candidate_ids = (
            generated_ids + [token_id]
        )

        candidate_text = TOKENIZER.decode(
            candidate_ids,
            skip_special_tokens=True,
        )

        candidate_word_count = _count_words(
            candidate_text
        )

        if candidate_word_count > current_word_count:

            if score > best_score:
                best_score = score
                best_token = token_id

    return best_token


# =========================================================================
# MAIN GENERATION FUNCTION
# =========================================================================

@torch.no_grad()
def generate_architecture_A_continuation(
    prompt,
    target_words,
    max_new_tokens,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):
    """
    Generate an exact number of whitespace-separated continuation words.

    FAST:
        Uses word-boundary rescue when the model gets stuck.

    PRODUCTION:
        Normal sampling is used unless word generation stalls.
    """

    assert isinstance(prompt, str)
    assert prompt.strip()

    assert target_words > 0
    assert max_new_tokens > 0

    MODEL.eval()

    # -------------------------------------------------------------
    # Encode prompt
    # -------------------------------------------------------------
    encoded_prompt = TOKENIZER.encode(prompt)

    prompt_ids = list(encoded_prompt.ids)

    assert len(prompt_ids) > 0

    # -------------------------------------------------------------
    # Context handling
    # -------------------------------------------------------------
    if len(prompt_ids) > CONTEXT_LENGTH:

        if RUN_MODE == "fast":

            prompt_ids = (
                prompt_ids[-CONTEXT_LENGTH:]
            )

        else:

            raise ValueError(
                "Production prompt exceeds "
                f"context length: {len(prompt_ids)} > "
                f"{CONTEXT_LENGTH}"
            )

    # -------------------------------------------------------------
    # Generated continuation tokens only
    # -------------------------------------------------------------
    generated_ids = []

    # Number of consecutive generated tokens that failed
    # to increase the whitespace-word count.
    stalled_tokens = 0

    last_word_count = 0

    # -------------------------------------------------------------
    # Autoregressive generation
    # -------------------------------------------------------------
    for step in range(max_new_tokens):

        current_ids = (
            prompt_ids + generated_ids
        )

        # Keep model input inside context window.
        if len(current_ids) > CONTEXT_LENGTH:

            current_ids = (
                current_ids[-CONTEXT_LENGTH:]
            )

        input_tensor = torch.tensor(
            [current_ids],
            dtype=torch.long,
            device=DEVICE,
        )

        # ---------------------------------------------------------
        # Forward pass
        # ---------------------------------------------------------
        with autocast_context():

            logits = _architecture_A_extract_logits(
                        model_output
                    )

            next_logits = logits[
                            0,
                            -1,
            ]

        # ---------------------------------------------------------
        # Current continuation word count
        # ---------------------------------------------------------
        current_text = TOKENIZER.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        current_word_count = _count_words(
            current_text
        )

        # ---------------------------------------------------------
        # Word-boundary rescue
        # ---------------------------------------------------------
        use_rescue = (
            RUN_MODE == "fast"
            and stalled_tokens >= 16
            and current_word_count < target_words
        )

        if use_rescue:

            rescue_token = (
                _find_word_advancing_token(
                    next_logits,
                    generated_ids,
                    current_word_count,
                    top_k_candidates=100,
                )
            )

            if rescue_token is not None:

                next_token = rescue_token

            else:

                next_token = _sample_next_token(
                    next_logits,
                    generated_ids,
                    temperature,
                    top_p,
                    top_k,
                    repetition_penalty,
                )

        else:

            next_token = _sample_next_token(
                next_logits,
                generated_ids,
                temperature,
                top_p,
                top_k,
                repetition_penalty,
            )

        generated_ids.append(next_token)

        # ---------------------------------------------------------
        # Check progress
        # ---------------------------------------------------------
        updated_text = TOKENIZER.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        updated_word_count = _count_words(
            updated_text
        )

        if updated_word_count > last_word_count:

            stalled_tokens = 0

        else:

            stalled_tokens += 1

        last_word_count = updated_word_count

        # ---------------------------------------------------------
        # Exact target reached
        # ---------------------------------------------------------
        if updated_word_count >= target_words:

            break

    # =========================================================================
    # FINAL TEXT
    # =========================================================================

    final_text = TOKENIZER.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    final_words = final_text.split()

    # -------------------------------------------------------------
    # Exact word count
    # -------------------------------------------------------------
    if len(final_words) >= target_words:

        final_text = " ".join(
            final_words[:target_words]
        )

    final_word_count = _count_words(
        final_text
    )

    # -------------------------------------------------------------
    # FAST failure message
    # -------------------------------------------------------------
    assert (
        final_word_count == target_words
    ), (
        "Architecture-A could not produce the "
        f"required {target_words} words within "
        f"{max_new_tokens} generated tokens. "
        f"Produced {final_word_count} words."
    )

    return final_text


# =========================================================================
# COMPATIBILITY ALIASES
# =========================================================================

generate_continuation_A = (
    generate_architecture_A_continuation
)

generate_continuation = (
    generate_architecture_A_continuation
)

generate_text = (
    generate_architecture_A_continuation
)


print(
    "✅ Architecture-A robust generator ready."
)

print(
    f"Mode                 : {RUN_MODE}"
)

print(
    f"Target words         : "
    f"{GENERATION_CONFIG['target_words']}"
)

print(
    f"Maximum tokens       : "
    f"{GENERATION_CONFIG['max_new_tokens']}"
)

print(
    f"Word-boundary rescue : "
    f"{RUN_MODE == 'fast'}"
)

print(
    f"Special tokens blocked: "
    f"{len(SPECIAL_TOKEN_IDS_A)}"
)

✅ Architecture-A robust generator ready.
Mode                 : production
Target words         : 128
Maximum tokens       : 1536
Word-boundary rescue : False
Special tokens blocked: 0


In [14]:
# Cell 12.5 — Fix Architecture-A tokenizer alias
# DECISION: ADD

assert "tokenizer" in globals(), (
    "❌ Lowercase tokenizer is not available. "
    "Run the tokenizer setup cell first."
)

# The Architecture-A generation helper expects TOKENIZER,
# while the notebook stores the tokenizer as `tokenizer`.
TOKENIZER = tokenizer

print("✅ Architecture-A tokenizer alias fixed.")
print(f"Tokenizer type: {type(TOKENIZER)}")

✅ Architecture-A tokenizer alias fixed.
Tokenizer type: <class 'tokenizers.Tokenizer'>


In [15]:
# Cell 12.6 — Fix Architecture-A context-length alias
# DECISION: ADD

assert "MODEL_CONFIG" in globals(), (
    "❌ MODEL_CONFIG is not available. "
    "Run the Architecture-A model/configuration cells first."
)

assert "context_length" in MODEL_CONFIG, (
    "❌ MODEL_CONFIG does not contain 'context_length'."
)

CONTEXT_LENGTH = int(
    MODEL_CONFIG["context_length"]
)

assert CONTEXT_LENGTH > 0

print("✅ Architecture-A context-length alias fixed.")
print(
    f"Context length: {CONTEXT_LENGTH:,}"
)

✅ Architecture-A context-length alias fixed.
Context length: 2,048


In [16]:
# ============================================================================
# Cell 12.7 — FINAL Architecture-A generation helper
# DECISION: REPLACE
#
# Designed specifically for the shared tokenizer from Cell 03:
#     tokenizer.encode(text).ids
#     tokenizer.decode(ids, skip_special_tokens=True)
#
# Guarantees:
#   - Correct MODEL(...) forward call
#   - Correct tuple/tensor output handling
#   - No hidden/stale model_output dependency
#   - No hidden CONTEXT_LENGTH dependency
#   - Uses lowercase `tokenizer`
#   - Exact 128-word output
#   - Strong word-boundary rescue for BPE-heavy/collapsed generations
#   - Inference-only changes; model weights are untouched
# ============================================================================

import math
from contextlib import nullcontext

import torch
import torch.nn.functional as F


print(
    "✍️ Building FINAL Architecture-A generation helper..."
)


# ============================================================================
# 1. REQUIRED GLOBALS
# ============================================================================

assert "MODEL" in globals(), (
    "❌ MODEL is unavailable."
)

assert "DEVICE" in globals(), (
    "❌ DEVICE is unavailable."
)

assert "MODEL_CONFIG" in globals(), (
    "❌ MODEL_CONFIG is unavailable."
)

assert "GENERATION_CONFIG" in globals(), (
    "❌ GENERATION_CONFIG is unavailable."
)

assert "tokenizer" in globals(), (
    "❌ Shared lowercase `tokenizer` from Cell 03 is unavailable.\n"
    "Run Cell 03 first."
)


# ============================================================================
# 2. CANONICAL TOKENIZER
# ============================================================================

# Cell 03 defines the shared tokenizer as lowercase `tokenizer`.
# Architecture-A uses that exact tokenizer.

A_TOKENIZER = tokenizer

assert hasattr(
    A_TOKENIZER,
    "encode",
), (
    "❌ Shared tokenizer does not provide encode()."
)

assert hasattr(
    A_TOKENIZER,
    "decode",
), (
    "❌ Shared tokenizer does not provide decode()."
)


# Preserve compatibility with any later cells that still reference TOKENIZER.
TOKENIZER = A_TOKENIZER


# ============================================================================
# 3. CONTEXT LENGTH
# ============================================================================

A_CONTEXT_LENGTH = int(
    MODEL_CONFIG["context_length"]
)

assert A_CONTEXT_LENGTH > 0, (
    "❌ Invalid Architecture-A context length."
)


# ============================================================================
# 4. LOCAL AUTOCAST
# ============================================================================

def _architecture_A_autocast():

    helper = globals().get(
        "autocast_context"
    )

    if callable(helper):
        return helper()

    return nullcontext()


# ============================================================================
# 5. WORD COUNT
# ============================================================================

def _architecture_A_count_words(
    text
):
    """
    Official Phase-1 convention:
    whitespace-separated words.
    """

    return len(
        str(text)
        .strip()
        .split()
    )


# Compatibility for Cell 14.
count_words = (
    _architecture_A_count_words
)


# ============================================================================
# 6. SAFE TOKENIZER ENCODE
# ============================================================================

def _architecture_A_encode(
    text
):
    """
    Supports the exact tokenizers.Tokenizer interface from Cell 03.
    """

    encoded = (
        A_TOKENIZER.encode(
            str(text)
        )
    )

    if hasattr(
        encoded,
        "ids",
    ):
        ids = list(
            encoded.ids
        )
    else:
        ids = list(
            encoded
        )

    return [
        int(x)
        for x in ids
    ]


# ============================================================================
# 7. SAFE TOKENIZER DECODE
# ============================================================================

def _architecture_A_decode(
    token_ids
):
    """
    Decode while tolerating minor tokenizer API differences.
    """

    token_ids = [
        int(x)
        for x in token_ids
    ]

    try:

        return str(
            A_TOKENIZER.decode(
                token_ids,
                skip_special_tokens=True,
            )
        )

    except TypeError:

        return str(
            A_TOKENIZER.decode(
                token_ids
            )
        )


# ============================================================================
# 8. MODEL OUTPUT EXTRACTION
# ============================================================================

def _architecture_A_extract_logits(
    model_output
):
    """
    Architecture-A training code uses:

        _, loss = MODEL(..., targets=targets)

    Therefore MODEL normally returns:

        (logits, loss)

    This helper also supports a direct logits tensor.
    """

    # Direct tensor.
    if torch.is_tensor(
        model_output
    ):

        logits = model_output

    # Tuple/list.
    elif isinstance(
        model_output,
        (tuple, list),
    ):

        assert len(
            model_output
        ) > 0, (
            "❌ MODEL returned an empty tuple/list."
        )

        logits = (
            model_output[0]
        )

    # Dictionary-like output.
    elif isinstance(
        model_output,
        dict,
    ):

        logits = None

        for key in (
            "logits",
            "lm_logits",
            "output",
        ):

            if key in model_output:

                logits = (
                    model_output[key]
                )

                break

        assert logits is not None, (
            "❌ MODEL returned a dict without logits."
        )

    else:

        raise TypeError(
            "❌ Unsupported MODEL output type: "
            f"{type(model_output)}"
        )

    assert torch.is_tensor(
        logits
    ), (
        "❌ Extracted Architecture-A logits "
        "are not a tensor."
    )

    assert logits.ndim == 3, (
        "❌ Expected logits with shape "
        "(batch, sequence, vocabulary).\n"
        f"Actual shape: {tuple(logits.shape)}"
    )

    return logits


# ============================================================================
# 9. CORRECT MODEL FORWARD
# ============================================================================

def _architecture_A_forward(
    input_tensor
):
    """
    IMPORTANT:
    MODEL(...) is explicitly called here.

    This eliminates the stale/undefined `model_output` problem.
    """

    MODEL.eval()

    # ------------------------------------------------------------------------
    # Primary inference call.
    # ------------------------------------------------------------------------

    try:

        with _architecture_A_autocast():

            model_output = (
                MODEL(
                    input_tensor
                )
            )

    except TypeError as first_error:

        # --------------------------------------------------------------------
        # Some implementations expose `targets` explicitly.
        # --------------------------------------------------------------------

        try:

            with _architecture_A_autocast():

                model_output = (
                    MODEL(
                        input_tensor,
                        targets=None,
                    )
                )

        except TypeError as second_error:

            raise RuntimeError(
                "❌ Architecture-A MODEL could not "
                "be called for inference.\n\n"
                f"MODEL(input_tensor):\n"
                f"{first_error}\n\n"
                f"MODEL(input_tensor, targets=None):\n"
                f"{second_error}"
            ) from second_error

    return _architecture_A_extract_logits(
        model_output
    )


# ============================================================================
# 10. SPECIAL TOKEN DISCOVERY
# ============================================================================

def _architecture_A_get_token_id(
    token_text
):

    try:

        token_id = (
            A_TOKENIZER.token_to_id(
                token_text
            )
        )

        if token_id is not None:

            return int(
                token_id
            )

    except Exception:

        pass

    return None


SPECIAL_TOKEN_IDS_A = set()


for token_text in (
    "<unk>",
    "<bos>",
    "<eos>",
    "<pad>",
):

    token_id = (
        _architecture_A_get_token_id(
            token_text
        )
    )

    if token_id is not None:

        SPECIAL_TOKEN_IDS_A.add(
            token_id
        )


# Also respect notebook-global token IDs when present.
for global_name in (
    "UNK_ID",
    "BOS_ID",
    "EOS_ID",
    "PAD_ID",
    "UNK_TOKEN_ID",
    "BOS_TOKEN_ID",
    "EOS_TOKEN_ID",
    "PAD_TOKEN_ID",
):

    if global_name in globals():

        value = globals()[
            global_name
        ]

        if value is not None:

            try:

                SPECIAL_TOKEN_IDS_A.add(
                    int(value)
                )

            except Exception:

                pass


# ============================================================================
# 11. REPETITION PENALTY
# ============================================================================

def _architecture_A_apply_repetition_penalty(
    logits,
    generated_ids,
    penalty,
):
    penalty = float(
        penalty
    )

    adjusted = (
        logits.float().clone()
    )

    if (
        penalty <= 1.0
        or not generated_ids
    ):

        return adjusted

    for token_id in set(
        int(x)
        for x in generated_ids
    ):

        if (
            token_id < 0
            or token_id >= adjusted.numel()
        ):
            continue

        value = adjusted[
            token_id
        ]

        if value < 0:

            adjusted[
                token_id
            ] = value * penalty

        else:

            adjusted[
                token_id
            ] = value / penalty

    return adjusted


# ============================================================================
# 12. TOP-P FILTER
# ============================================================================

def _architecture_A_top_p_filter(
    logits,
    top_p,
):
    top_p = float(
        top_p
    )

    if (
        top_p >= 1.0
        or top_p <= 0.0
    ):

        return logits

    sorted_logits, sorted_indices = (
        torch.sort(
            logits,
            descending=True,
        )
    )

    sorted_probs = F.softmax(
        sorted_logits,
        dim=-1,
    )

    cumulative_probs = (
        torch.cumsum(
            sorted_probs,
            dim=-1,
        )
    )

    remove_mask = (
        cumulative_probs > top_p
    )

    # Keep the first token crossing the threshold.
    remove_mask[1:] = (
        remove_mask[:-1].clone()
    )

    remove_mask[0] = False

    sorted_logits = (
        sorted_logits.masked_fill(
            remove_mask,
            -float("inf"),
        )
    )

    filtered = torch.full_like(
        logits,
        -float("inf"),
    )

    filtered.scatter_(
        0,
        sorted_indices,
        sorted_logits,
    )

    return filtered


# ============================================================================
# 13. SAMPLE TOKEN
# ============================================================================

def _architecture_A_sample_token(
    logits,
    generated_ids,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):
    logits = (
        _architecture_A_apply_repetition_penalty(
            logits,
            generated_ids,
            repetition_penalty,
        )
    )

    # ------------------------------------------------------------------------
    # Block structural tokens.
    # ------------------------------------------------------------------------

    for token_id in (
        SPECIAL_TOKEN_IDS_A
    ):

        if (
            0 <= token_id
            < logits.numel()
        ):

            logits[
                token_id
            ] = -float("inf")

    # ------------------------------------------------------------------------
    # Temperature.
    # ------------------------------------------------------------------------

    temperature = max(
        float(temperature),
        1e-5,
    )

    logits = (
        logits / temperature
    )

    # ------------------------------------------------------------------------
    # Top-k.
    # ------------------------------------------------------------------------

    if (
        top_k > 0
        and top_k < logits.numel()
    ):

        k = min(
            int(top_k),
            logits.numel(),
        )

        values, indices = (
            torch.topk(
                logits,
                k=k,
            )
        )

        filtered = torch.full_like(
            logits,
            -float("inf"),
        )

        filtered[
            indices
        ] = values

        logits = filtered

    # ------------------------------------------------------------------------
    # Top-p.
    # ------------------------------------------------------------------------

    logits = (
        _architecture_A_top_p_filter(
            logits,
            top_p,
        )
    )

    probs = F.softmax(
        logits,
        dim=-1,
    )

    # Defensive fallback.
    if (
        not torch.isfinite(
            probs
        ).all()
        or float(
            probs.sum().item()
        ) <= 0.0
    ):

        valid_logits = (
            logits.clone()
        )

        for token_id in (
            SPECIAL_TOKEN_IDS_A
        ):

            if (
                0 <= token_id
                < valid_logits.numel()
            ):

                valid_logits[
                    token_id
                ] = -float("inf")

        return int(
            torch.argmax(
                valid_logits
            ).item()
        )

    return int(
        torch.multinomial(
            probs,
            num_samples=1,
        ).item()
    )


# ============================================================================
# 14. WORD-ADVANCING RESCUE
# ============================================================================

def _architecture_A_find_word_advancing_token(
    logits,
    generated_ids,
    current_word_count,
    candidate_k=1024,
):
    """
    Model-derived rescue.

    We inspect high-probability vocabulary candidates and select a token
    whose addition actually increases the whitespace-separated word count.

    This is particularly important for BPE sequences that can spend many
    tokens inside a single word.
    """

    candidate_logits = (
        logits.float().clone()
    )

    for token_id in (
        SPECIAL_TOKEN_IDS_A
    ):

        if (
            0 <= token_id
            < candidate_logits.numel()
        ):

            candidate_logits[
                token_id
            ] = -float("inf")

    candidate_k = min(
        int(candidate_k),
        candidate_logits.numel(),
    )

    if candidate_k <= 0:
        return None

    candidate_values, candidate_indices = (
        torch.topk(
            candidate_logits,
            k=candidate_k,
        )
    )

    # Decode a limited recent suffix.
    # This greatly reduces rescue overhead while preserving local word
    # boundary information.
    recent_ids = generated_ids[
        -128:
    ]

    try:

        current_suffix_text = (
            _architecture_A_decode(
                recent_ids
            )
        )

    except Exception:

        current_suffix_text = ""

    # The suffix count can differ from the full count by a constant offset,
    # so we compare candidate suffix word counts against the current suffix.
    suffix_base_count = (
        _architecture_A_count_words(
            current_suffix_text
        )
    )

    best_token = None
    best_score = -float("inf")

    for score, token_id in zip(
        candidate_values.tolist(),
        candidate_indices.tolist(),
    ):

        if not math.isfinite(
            float(score)
        ):

            continue

        token_id = int(
            token_id
        )

        candidate_suffix = (
            recent_ids
            + [token_id]
        )

        try:

            candidate_text = (
                _architecture_A_decode(
                    candidate_suffix
                )
            )

        except Exception:

            continue

        candidate_count = (
            _architecture_A_count_words(
                candidate_text
            )
        )

        if (
            candidate_count
            > suffix_base_count
        ):

            if (
                float(score)
                > best_score
            ):

                best_score = float(
                    score
                )

                best_token = token_id

    return best_token


# ============================================================================
# 15. MAIN GENERATION FUNCTION
# ============================================================================

@torch.inference_mode()
def generate_architecture_A_continuation(
    prompt,
    target_words,
    max_new_tokens,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):
    """
    Generate exactly `target_words` whitespace-separated words.

    The returned value contains ONLY the continuation.
    """

    # ------------------------------------------------------------------------
    # Normalize arguments.
    # ------------------------------------------------------------------------

    prompt = str(
        prompt
    ).strip()

    target_words = int(
        target_words
    )

    requested_max_tokens = int(
        max_new_tokens
    )

    temperature = float(
        temperature
    )

    top_p = float(
        top_p
    )

    top_k = int(
        top_k
    )

    repetition_penalty = float(
        repetition_penalty
    )

    assert prompt, (
        "❌ Empty generation prompt."
    )

    assert target_words > 0, (
        "❌ target_words must be positive."
    )

    assert requested_max_tokens > 0, (
        "❌ max_new_tokens must be positive."
    )


    # ------------------------------------------------------------------------
    # Encode prompt using shared tokenizer.
    # ------------------------------------------------------------------------

    prompt_ids = (
        _architecture_A_encode(
            prompt
        )
    )

    assert prompt_ids, (
        "❌ Prompt encoded to zero tokens."
    )


    # ------------------------------------------------------------------------
    # Keep prompt within model context.
    # ------------------------------------------------------------------------

    if len(
        prompt_ids
    ) > A_CONTEXT_LENGTH:

        prompt_ids = (
            prompt_ids[
                -A_CONTEXT_LENGTH:
            ]
        )


    # ------------------------------------------------------------------------
    # Generated IDs only.
    # ------------------------------------------------------------------------

    generated_ids = []


    # ------------------------------------------------------------------------
    # Safety generation budget.
    #
    # Previous run:
    #     1536 tokens -> 18 words
    #
    # Therefore the configured cap alone is not sufficient.
    #
    # The rescue logic below is designed to prevent the same long
    # subword stall from continuing indefinitely.
    # ------------------------------------------------------------------------

    safety_budget = max(
        requested_max_tokens,
        target_words * 16,
    )

    safety_budget = min(
        safety_budget,
        4096,
    )


    # ------------------------------------------------------------------------
    # Stall tracking.
    # ------------------------------------------------------------------------

    stalled_tokens = 0
    last_word_count = 0


    # =========================================================================
    # AUTOREGRESSIVE LOOP
    # =========================================================================

    for step in range(
        safety_budget
    ):

        # ---------------------------------------------------------------------
        # Construct current rolling context.
        # ---------------------------------------------------------------------

        current_ids = (
            prompt_ids
            + generated_ids
        )

        if len(
            current_ids
        ) > A_CONTEXT_LENGTH:

            current_ids = (
                current_ids[
                    -A_CONTEXT_LENGTH:
                ]
            )


        input_tensor = torch.tensor(
            [current_ids],
            dtype=torch.long,
            device=DEVICE,
        )


        # ---------------------------------------------------------------------
        # CORRECT MODEL FORWARD
        # ---------------------------------------------------------------------

        logits = (
            _architecture_A_forward(
                input_tensor
            )
        )


        # ---------------------------------------------------------------------
        # Last-position logits.
        # ---------------------------------------------------------------------

        next_logits = (
            logits[
                0,
                -1,
                :
            ]
            .float()
        )


        # ---------------------------------------------------------------------
        # Current word count.
        # ---------------------------------------------------------------------

        current_text = (
            _architecture_A_decode(
                generated_ids
            )
        )

        current_word_count = (
            _architecture_A_count_words(
                current_text
            )
        )


        # ---------------------------------------------------------------------
        # Word-boundary rescue.
        #
        # Normal generation:
        #     start rescue after 8 stalled tokens.
        #
        # Strong rescue:
        #     every subsequent 8 stalled tokens.
        # ---------------------------------------------------------------------

        use_rescue = (
            stalled_tokens >= 8
            and current_word_count
            < target_words
        )


        if use_rescue:

            rescue_token = (
                _architecture_A_find_word_advancing_token(
                    next_logits,
                    generated_ids,
                    current_word_count,
                    candidate_k=1024,
                )
            )

            if rescue_token is not None:

                next_token = (
                    rescue_token
                )

            else:

                next_token = (
                    _architecture_A_sample_token(
                        next_logits,
                        generated_ids,
                        temperature,
                        top_p,
                        top_k,
                        repetition_penalty,
                    )
                )

        else:

            next_token = (
                _architecture_A_sample_token(
                    next_logits,
                    generated_ids,
                    temperature,
                    top_p,
                    top_k,
                    repetition_penalty,
                )
            )


        # ---------------------------------------------------------------------
        # Append token.
        # ---------------------------------------------------------------------

        generated_ids.append(
            int(next_token)
        )


        # ---------------------------------------------------------------------
        # Measure word progress.
        # ---------------------------------------------------------------------

        updated_text = (
            _architecture_A_decode(
                generated_ids
            )
        )

        updated_word_count = (
            _architecture_A_count_words(
                updated_text
            )
        )


        if (
            updated_word_count
            > last_word_count
        ):

            stalled_tokens = 0

        else:

            stalled_tokens += 1


        last_word_count = (
            updated_word_count
        )


        # ---------------------------------------------------------------------
        # Target reached.
        # ---------------------------------------------------------------------

        if (
            updated_word_count
            >= target_words
        ):

            break


    # =========================================================================
    # FINAL DECODE
    # =========================================================================

    final_text = (
        _architecture_A_decode(
            generated_ids
        )
        .strip()
    )


    final_words = (
        final_text
        .split()
    )


    # Keep exactly target_words.
    if len(
        final_words
    ) >= target_words:

        final_text = " ".join(
            final_words[
                :target_words
            ]
        )


    final_word_count = (
        _architecture_A_count_words(
            final_text
        )
    )


    # =========================================================================
    # FINAL CHECK
    # =========================================================================

    if (
        final_word_count
        != target_words
    ):

        raise RuntimeError(
            "\n❌ Architecture-A could not "
            "reach the required word count.\n\n"
            f"Target words        : {target_words}\n"
            f"Produced words      : {final_word_count}\n"
            f"Generated tokens    : {len(generated_ids)}\n"
            f"Requested token cap : {requested_max_tokens}\n"
            f"Safety token cap    : {safety_budget}\n"
            f"Context length      : {A_CONTEXT_LENGTH}\n"
            f"Last word count     : {last_word_count}\n"
            f"Stalled tokens      : {stalled_tokens}\n"
            "\n"
            "This is now a genuine generation/model/tokenizer "
            "diagnostic rather than a hidden NameError."
        )


    assert final_text, (
        "❌ Architecture-A produced empty text."
    )


    return final_text


# ============================================================================
# 16. COMPATIBILITY ALIASES
# ============================================================================

generate_continuation_A = (
    generate_architecture_A_continuation
)

generate_continuation = (
    generate_architecture_A_continuation
)

generate_text = (
    generate_architecture_A_continuation
)


# ============================================================================
# 17. PRE-FLIGHT MODEL CHECK
# ============================================================================

_TEST_TEXT = (
    "The court considered the matter"
)

_TEST_IDS = (
    _architecture_A_encode(
        _TEST_TEXT
    )
)

if len(
    _TEST_IDS
) > A_CONTEXT_LENGTH:

    _TEST_IDS = (
        _TEST_IDS[
            -A_CONTEXT_LENGTH:
        ]
    )

_TEST_TENSOR = torch.tensor(
    [_TEST_IDS],
    dtype=torch.long,
    device=DEVICE,
)

_TEST_LOGITS = (
    _architecture_A_forward(
        _TEST_TENSOR
    )
)

assert _TEST_LOGITS.shape[0] == 1, (
    f"❌ Unexpected batch dimension: "
    f"{_TEST_LOGITS.shape}"
)

assert _TEST_LOGITS.shape[1] == len(
    _TEST_IDS
), (
    "❌ Model sequence dimension does not "
    "match input sequence length."
)

assert _TEST_LOGITS.shape[2] == int(
    A_TOKENIZER.get_vocab_size()
), (
    "❌ Model vocabulary dimension does not "
    "match tokenizer vocabulary size.\n"
    f"Model: {int(_TEST_LOGITS.shape[2])}\n"
    f"Tokenizer: {int(A_TOKENIZER.get_vocab_size())}"
)


# ============================================================================
# 18. SUMMARY
# ============================================================================

print()
print(
    "✅ FINAL Architecture-A generation helper ready."
)

print(
    f"   Tokenizer vocab      : "
    f"{A_TOKENIZER.get_vocab_size():,}"
)

print(
    f"   Model vocab          : "
    f"{_TEST_LOGITS.shape[2]:,}"
)

print(
    f"   Context length       : "
    f"{A_CONTEXT_LENGTH}"
)

print(
    f"   Target words         : "
    f"{GENERATION_CONFIG['target_words']}"
)

print(
    f"   Configured max tok   : "
    f"{GENERATION_CONFIG['max_new_tokens']}"
)

print(
    f"   Inference safety cap : "
    f"{max(int(GENERATION_CONFIG['max_new_tokens']), int(GENERATION_CONFIG['target_words']) * 16)}"
)

print(
    f"   Special IDs blocked  : "
    f"{len(SPECIAL_TOKEN_IDS_A)}"
)

print(
    "   MODEL forward check  : PASSED"
)

print(
    "   Tokenizer check      : PASSED"
)

print(
    "✅ Ready for the official-test generation cell."
)

✍️ Building FINAL Architecture-A generation helper...

✅ FINAL Architecture-A generation helper ready.
   Tokenizer vocab      : 50,000
   Model vocab          : 50,000
   Context length       : 2048
   Target words         : 128
   Configured max tok   : 1536
   Inference safety cap : 2048
   Special IDs blocked  : 0
   MODEL forward check  : PASSED
   Tokenizer check      : PASSED
✅ Ready for the official-test generation cell.


In [17]:
# # Cell 12.7 — Normalize Architecture-A model output
# # DECISION: ADD
# #
# # The Architecture-A generation helper expects MODEL(...) to return
# # a logits tensor, but the current MODEL returns a tuple.
# # This helper extracts the logits tensor without changing the model
# # weights or checkpoint.

# def _architecture_A_extract_logits(
#     model_output
# ):
#     # -------------------------------------------------------------
#     # Case 1: MODEL returns the logits tensor directly.
#     # -------------------------------------------------------------
#     if torch.is_tensor(
#         model_output
#     ):
#         return model_output

#     # -------------------------------------------------------------
#     # Case 2: MODEL returns a tuple/list.
#     # The first element is expected to be logits.
#     # -------------------------------------------------------------
#     if isinstance(
#         model_output,
#         (tuple, list),
#     ):
#         assert len(model_output) > 0, (
#             "❌ Architecture-A MODEL returned "
#             "an empty tuple/list."
#         )

#         logits = model_output[0]

#         assert torch.is_tensor(
#             logits
#         ), (
#             "❌ First element returned by Architecture-A "
#             "MODEL is not a tensor."
#         )

#         return logits

#     raise TypeError(
#         "❌ Unexpected Architecture-A MODEL output type: "
#         f"{type(model_output)}"
#     )


# print(
#     "✅ Architecture-A model-output normalizer ready."
# )

In [18]:
# Cell 13 — Load and validate the official Phase-1 test set for Architecture-A
# DECISION: UPDATE

import json
import shutil
from pathlib import Path

print("📄 Loading Architecture-A official Phase-1 test set...")

# =========================================================================
# TEST DATA DIRECTORIES
# =========================================================================
TEST_DATA_DIR = (
    NOTEBOOK_DIR
    / "test_data"
)

TEST_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# =========================================================================
# OFFICIAL DATASET REPOSITORY
# =========================================================================
HF_REPO_ID = (
    "Exploration-Lab/CS-781-Capstone"
)

# =========================================================================
# SEARCH LOCAL KAGGLE PATHS FIRST
# =========================================================================
CANDIDATE_TEST_PATHS = [
    Path(
        "/kaggle/working/Phase_1_Pretraining/Test/test.json"
    ),

    Path(
        "/kaggle/working/Phase_1_Pretraining/Test/Phase1- Test.json"
    ),

    Path(
        "/kaggle/input/Phase_1_Pretraining/Test/test.json"
    ),

    Path(
        "/kaggle/input/Phase_1_Pretraining/Test/Phase1- Test.json"
    ),
]

TEST_JSON_PATH = None

for _candidate in CANDIDATE_TEST_PATHS:

    if _candidate.is_file():

        TEST_JSON_PATH = (
            _candidate
        )

        break

# =========================================================================
# BROADER LOCAL SEARCH
# =========================================================================
if TEST_JSON_PATH is None:

    SEARCH_ROOTS = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    _candidates = []

    for _root in SEARCH_ROOTS:

        if not _root.exists():
            continue

        try:

            _candidates.extend(
                _root.rglob(
                    "test.json"
                )
            )

            _candidates.extend(
                _root.rglob(
                    "*Phase1*Test*.json"
                )
            )

        except Exception:

            continue

    # -------------------------------------------------------------
    # Rank likely official Phase-1 test files first.
    # -------------------------------------------------------------
    def _test_path_rank(
        path
    ):

        normalized = (
            str(path)
            .replace("\\", "/")
            .lower()
        )

        if (
            "phase_1_pretraining/test/test.json"
            in normalized
        ):
            return 0

        if (
            "phase_1_pretraining"
            in normalized
        ):
            return 1

        if (
            "phase1"
            in normalized
            and "test"
            in normalized
        ):
            return 2

        if (
            Path(normalized).name
            == "test.json"
        ):
            return 3

        return 4

    _candidates = sorted(
        {
            path.resolve()
            for path in _candidates
            if path.is_file()
        },
        key=lambda path: (
            _test_path_rank(path),
            len(str(path)),
        ),
    )

    if _candidates:

        TEST_JSON_PATH = (
            _candidates[0]
        )

# =========================================================================
# OFFICIAL HF FALLBACK
# =========================================================================
if TEST_JSON_PATH is None:

    print(
        "ℹ️ Official test file not found locally."
    )

    print(
        "   Attempting official Hugging Face dataset."
    )

    from huggingface_hub import (
        HfApi,
        hf_hub_download,
    )

    # -------------------------------------------------------------
    # Retrieve HF token when available.
    # -------------------------------------------------------------
    HF_TOKEN = None

    try:

        from kaggle_secrets import (
            UserSecretsClient,
        )

        _secrets = (
            UserSecretsClient()
        )

        for _secret_name in (
            "HF_TOKEN",
            "HUGGINGFACE_TOKEN",
            "HUGGINGFACEHUB_API_TOKEN",
        ):

            try:

                _token = _secrets.get_secret(
                    _secret_name
                )

                if _token:

                    HF_TOKEN = _token

                    break

            except Exception:

                continue

    except Exception:

        pass

    _api = HfApi(
        token=HF_TOKEN
    )

    _repo_files = _api.list_repo_files(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
    )

    _normalized_files = {
        str(path).replace(
            "\\",
            "/",
        ): path
        for path in _repo_files
    }

    # -------------------------------------------------------------
    # Candidate official test paths.
    # -------------------------------------------------------------
    _test_candidates = []

    for _path in _repo_files:

        _normalized = (
            str(_path)
            .replace(
                "\\",
                "/",
            )
            .lower()
        )

        _basename = Path(
            _normalized
        ).name

        if (
            _basename == "test.json"
            or
            "phase1" in _normalized
            and "test" in _normalized
            and _basename.endswith(
                ".json"
            )
        ):

            _test_candidates.append(
                _path
            )

    assert _test_candidates, (
        "Could not locate the official Phase-1 "
        f"test JSON in {HF_REPO_ID}."
    )

    def _hf_test_rank(
        path
    ):

        normalized = (
            str(path)
            .replace("\\", "/")
            .lower()
        )

        basename = Path(
            normalized
        ).name

        if (
            normalized
            == "phase_1_pretraining/test/test.json"
        ):
            return 0

        if (
            "phase_1_pretraining"
            in normalized
            and basename == "test.json"
        ):
            return 1

        if basename == "test.json":
            return 2

        return 3

    _test_candidates = sorted(
        _test_candidates,
        key=_hf_test_rank,
    )

    DISCOVERED_TEST_REPO_PATH = (
        _test_candidates[0]
    )

    print(
        "✅ Discovered official test file:"
    )

    print(
        f"   {DISCOVERED_TEST_REPO_PATH}"
    )

    _downloaded = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=(
            DISCOVERED_TEST_REPO_PATH
        ),
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=str(
            TEST_DATA_DIR
        ),
    )

    TEST_JSON_PATH = (
        Path(_downloaded)
    )

# =========================================================================
# FINAL LOCAL COPY
# =========================================================================
assert TEST_JSON_PATH is not None

assert TEST_JSON_PATH.is_file(), (
    f"Official test file not found: "
    f"{TEST_JSON_PATH}"
)

A_TEST_TARGET = (
    TEST_DATA_DIR
    / "Phase_1_Pretraining"
    / "Test"
    / "test.json"
)

A_TEST_TARGET.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if (
    TEST_JSON_PATH.resolve()
    != A_TEST_TARGET.resolve()
):

    shutil.copy2(
        TEST_JSON_PATH,
        A_TEST_TARGET,
    )

TEST_JSON_PATH = (
    A_TEST_TARGET
)

print()
print(
    f"Official test file: "
    f"{TEST_JSON_PATH}"
)

# =========================================================================
# LOAD JSON
# =========================================================================
with open(
    TEST_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:

    TEST_JSON = json.load(
        f
    )

# =========================================================================
# NORMALIZE SUPPORTED STRUCTURES
# =========================================================================
if isinstance(
    TEST_JSON,
    list,
):

    OFFICIAL_TEST_RECORDS = (
        TEST_JSON
    )

elif isinstance(
    TEST_JSON,
    dict,
):

    OFFICIAL_TEST_RECORDS = None

    # -------------------------------------------------------------
    # List-based wrappers.
    # -------------------------------------------------------------
    for _key in (
        "data",
        "records",
        "test",
        "examples",
    ):

        _value = TEST_JSON.get(
            _key
        )

        if isinstance(
            _value,
            list,
        ):

            OFFICIAL_TEST_RECORDS = (
                _value
            )

            break

    # -------------------------------------------------------------
    # ID -> record mapping.
    # -------------------------------------------------------------
    if OFFICIAL_TEST_RECORDS is None:

        OFFICIAL_TEST_RECORDS = []

        for (
            _test_id,
            _value,
        ) in TEST_JSON.items():

            if isinstance(
                _value,
                dict,
            ):

                _record = dict(
                    _value
                )

                _record.setdefault(
                    "id",
                    str(_test_id),
                )

            else:

                _record = {
                    "id": str(
                        _test_id
                    ),
                    "prompt": str(
                        _value
                    ),
                }

            OFFICIAL_TEST_RECORDS.append(
                _record
            )

else:

    raise TypeError(
        "Unsupported official test JSON type: "
        f"{type(TEST_JSON)}"
    )

# =========================================================================
# FIELD RESOLUTION
# =========================================================================
def get_test_id(
    record
):

    for _key in (
        "id",
        "ID",
        "test_id",
        "testId",
    ):

        if _key in record:

            return str(
                record[_key]
            )

    raise KeyError(
        "No test ID field found. "
        f"Keys: {list(record.keys())}"
    )


def get_test_prompt(
    record
):

    for _key in (
        "prompt",
        "text",
        "input",
    ):

        if _key in record:

            return str(
                record[_key]
            )

    raise KeyError(
        "No test prompt field found. "
        f"Keys: {list(record.keys())}"
    )

# =========================================================================
# EXTRACT OFFICIAL TEST DATA
# =========================================================================
OFFICIAL_TEST_IDS = [
    get_test_id(
        record
    )
    for record in OFFICIAL_TEST_RECORDS
]

OFFICIAL_TEST_PROMPTS = [
    get_test_prompt(
        record
    )
    for record in OFFICIAL_TEST_RECORDS
]

assert len(
    OFFICIAL_TEST_RECORDS
) > 0

assert len(
    OFFICIAL_TEST_IDS
) == len(
    set(
        OFFICIAL_TEST_IDS
    )
), (
    "Duplicate official test IDs detected."
)

assert all(
    prompt.strip()
    for prompt in OFFICIAL_TEST_PROMPTS
), (
    "At least one official prompt is empty."
)

# =========================================================================
# OFFICIAL 512-WORD PROMPT VALIDATION
# =========================================================================
OFFICIAL_PROMPT_WORD_COUNTS = [
    len(
        prompt.strip().split()
    )
    for prompt in OFFICIAL_TEST_PROMPTS
]

assert all(
    count == 512
    for count in OFFICIAL_PROMPT_WORD_COUNTS
), (
    "Official Phase-1 test prompts must contain "
    "exactly 512 whitespace-separated words."
)

# =========================================================================
# SELECT ACTIVE TEST SET BY RUN MODE
# =========================================================================
if RUN_MODE == "fast":

    ACTIVE_TEST_RECORDS = (
        OFFICIAL_TEST_RECORDS[:2]
    )

else:

    ACTIVE_TEST_RECORDS = (
        OFFICIAL_TEST_RECORDS
    )

EXPECTED_TEST_COUNT = (
    2
    if RUN_MODE == "fast"
    else 1000
)

assert (
    len(ACTIVE_TEST_RECORDS)
    == EXPECTED_TEST_COUNT
), (
    f"Expected {EXPECTED_TEST_COUNT} active test records "
    f"for RUN_MODE={RUN_MODE}, found "
    f"{len(ACTIVE_TEST_RECORDS)}."
)

ACTIVE_TEST_IDS = [
    get_test_id(
        record
    )
    for record in ACTIVE_TEST_RECORDS
]

ACTIVE_TEST_PROMPTS = [
    get_test_prompt(
        record
    )
    for record in ACTIVE_TEST_RECORDS
]

# =========================================================================
# TEST MANIFEST
# =========================================================================
TEST_MANIFEST = {
    "project": "CS781 Phase 1",

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": RUN_MODE,

    "huggingface_repo": HF_REPO_ID,

    "test_json_path": str(
        TEST_JSON_PATH
    ),

    "total_official_examples": int(
        len(
            OFFICIAL_TEST_RECORDS
        )
    ),

    "active_examples": int(
        len(
            ACTIVE_TEST_RECORDS
        )
    ),

    "prompt_word_count": 512,

    "target_generation_words": int(
        GENERATION_CONFIG[
            "target_words"
        ]
    ),

    "active_test_ids": list(
        ACTIVE_TEST_IDS
    ),
}

TEST_MANIFEST_PATH = (
    CONFIGS_DIR
    / "official_test_manifest_A.json"
)

with open(
    TEST_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TEST_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =========================================================================
# SUMMARY
# =========================================================================
print()
print(
    "✅ Official Phase-1 test set loaded."
)

print(
    f"Official examples : "
    f"{len(OFFICIAL_TEST_RECORDS):,}"
)

print(
    f"Active examples   : "
    f"{len(ACTIVE_TEST_RECORDS):,}"
)

print(
    f"Prompt words      : "
    f"{min(OFFICIAL_PROMPT_WORD_COUNTS)}"
    f"-"
    f"{max(OFFICIAL_PROMPT_WORD_COUNTS)}"
)

print(
    f"Target generation : "
    f"{GENERATION_CONFIG['target_words']} words"
)

print(
    f"Test manifest     : "
    f"{TEST_MANIFEST_PATH}"
)

📄 Loading Architecture-A official Phase-1 test set...

Official test file: /kaggle/working/architecture_A/test_data/Phase_1_Pretraining/Test/test.json

✅ Official Phase-1 test set loaded.
Official examples : 1,000
Active examples   : 1,000
Prompt words      : 512-512
Target generation : 128 words
Test manifest     : /kaggle/working/architecture_A/configs/official_test_manifest_A.json


In [19]:
# ============================================================================
# Cell 13.9 — Architecture-A generation compatibility helpers
# DECISION: ADD
# PURPOSE: Prevent undefined-helper errors during official-test generation.
# ============================================================================

def count_words(text):
    """
    Count whitespace-separated words.

    This matches the official Phase-1 style of checking generated
    continuation length: one word = one whitespace-separated token.
    """
    if text is None:
        return 0

    text = str(text).strip()

    if not text:
        return 0

    return len(text.split())


# ---------------------------------------------------------------------------
# Preflight checks
# ---------------------------------------------------------------------------

assert callable(count_words), (
    "count_words helper was not created correctly."
)

assert "tokenizer" in globals(), (
    "Tokenizer is unavailable. Run the tokenizer/setup cell first."
)

assert "MODEL" in globals(), (
    "MODEL is unavailable. Run the Architecture-A model setup cell first."
)

assert "MODEL_CONFIG" in globals(), (
    "MODEL_CONFIG is unavailable."
)

assert "GENERATION_CONFIG" in globals(), (
    "GENERATION_CONFIG is unavailable."
)

assert "ACTIVE_TEST_RECORDS" in globals(), (
    "ACTIVE_TEST_RECORDS is unavailable. Run the official-test loading cell."
)

assert "FINAL_MODEL_PATH" in globals(), (
    "FINAL_MODEL_PATH is unavailable."
)

assert "generate_architecture_A_continuation" in globals(), (
    "generate_architecture_A_continuation is unavailable. "
    "Run the corrected Architecture-A generation helper cell first."
)

print("✅ Architecture-A generation compatibility checks passed.")
print(f"   Active test records : {len(ACTIVE_TEST_RECORDS):,}")
print(f"   Context length      : {MODEL_CONFIG['context_length']}")
print(f"   Target words        : {GENERATION_CONFIG['target_words']}")
print(f"   Max new tokens      : {GENERATION_CONFIG['max_new_tokens']}")

✅ Architecture-A generation compatibility checks passed.
   Active test records : 1,000
   Context length      : 2048
   Target words        : 128
   Max new tokens      : 1536


In [20]:
# Cell 13.5 — Restore Architecture-A generation checkpoint from Kaggle Dataset
# DECISION: ADD

from pathlib import Path
import shutil

print("♻️ Checking for saved Architecture-A generation checkpoint...")

WORKING_ROOT = Path("/kaggle/working/architecture_A")
WORKING_PREDICTIONS = WORKING_ROOT / "predictions"
WORKING_METRICS = WORKING_ROOT / "metrics"

WORKING_PREDICTIONS.mkdir(parents=True, exist_ok=True)
WORKING_METRICS.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# Find saved prediction_A_partial.json anywhere under /kaggle/input
# -------------------------------------------------------------------------
partial_candidates = list(
    Path("/kaggle/input").rglob("prediction_A_partial.json")
)

progress_candidates = list(
    Path("/kaggle/input").rglob("generation_progress_A.json")
)

print(f"Found partial files   : {len(partial_candidates)}")
print(f"Found progress files  : {len(progress_candidates)}")

# -------------------------------------------------------------------------
# Restore partial predictions
# -------------------------------------------------------------------------
if partial_candidates:
    source_partial = partial_candidates[0]

    destination_partial = (
        WORKING_PREDICTIONS / "prediction_A_partial.json"
    )

    shutil.copy2(
        source_partial,
        destination_partial,
    )

    print(f"✅ Restored partial predictions:")
    print(f"   From : {source_partial}")
    print(f"   To   : {destination_partial}")

else:
    print("⚠️ No prediction_A_partial.json found in /kaggle/input.")

# -------------------------------------------------------------------------
# Restore generation progress
# -------------------------------------------------------------------------
if progress_candidates:
    source_progress = progress_candidates[0]

    destination_progress = (
        WORKING_METRICS / "generation_progress_A.json"
    )

    shutil.copy2(
        source_progress,
        destination_progress,
    )

    print(f"✅ Restored generation progress:")
    print(f"   From : {source_progress}")
    print(f"   To   : {destination_progress}")

else:
    print(
        "⚠️ No generation_progress_A.json found in /kaggle/input."
    )

# -------------------------------------------------------------------------
# Show what will be available to Cell 14
# -------------------------------------------------------------------------
print()
print("Working checkpoint status:")

print(
    "   Partial :",
    (
        "✅"
        if (WORKING_PREDICTIONS / "prediction_A_partial.json").is_file()
        else "❌"
    ),
)

print(
    "   Progress:",
    (
        "✅"
        if (WORKING_METRICS / "generation_progress_A.json").is_file()
        else "❌"
    ),
)

# -------------------------------------------------------------------------
# If partial exists, inspect number of saved predictions
# -------------------------------------------------------------------------
partial_path = (
    WORKING_PREDICTIONS / "prediction_A_partial.json"
)

if partial_path.is_file():
    import json

    with open(
        partial_path,
        "r",
        encoding="utf-8",
    ) as f:
        saved_predictions = json.load(f)

    print()
    print(
        f"✅ Saved predictions available: "
        f"{len(saved_predictions):,}"
    )

    if saved_predictions:
        saved_ids = list(saved_predictions.keys())
        print(
            f"   First saved ID : {saved_ids[0]}"
        )
        print(
            f"   Last saved ID  : {saved_ids[-1]}"
        )

♻️ Checking for saved Architecture-A generation checkpoint...
Found partial files   : 1
Found progress files  : 1
✅ Restored partial predictions:
   From : /kaggle/input/datasets/rohitbatra2004/phase111/architecture_A/predictions/prediction_A_partial.json
   To   : /kaggle/working/architecture_A/predictions/prediction_A_partial.json
✅ Restored generation progress:
   From : /kaggle/input/datasets/rohitbatra2004/phase111/architecture_A/metrics/generation_progress_A.json
   To   : /kaggle/working/architecture_A/metrics/generation_progress_A.json

Working checkpoint status:
   Partial : ✅
   Progress: ✅

✅ Saved predictions available: 1,000
   First saved ID : 1
   Last saved ID  : 1000


In [21]:
# Cell 14 — Generate Architecture-A official-test continuations with automatic resume
# DECISION: ADD

import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch

print("🚀 Starting Architecture-A official-test generation...")

# =========================================================================
# DIRECTORIES
# =========================================================================
NOTEBOOK_DIR = Path(
    "/kaggle/working/architecture_A"
)

METRICS_DIR = NOTEBOOK_DIR / "metrics"
PREDICTIONS_DIR = NOTEBOOK_DIR / "predictions"

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# =========================================================================
# REQUIRED MODEL
# =========================================================================
assert FINAL_MODEL_PATH.is_file(), (
    f"Final Architecture-A model not found: "
    f"{FINAL_MODEL_PATH}"
)

# -------------------------------------------------------------------------
# Load the selected best checkpoint.
# -------------------------------------------------------------------------
BEST_CHECKPOINT = torch.load(
    FINAL_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint_is_compatible(
    BEST_CHECKPOINT
), (
    "Selected Architecture-A checkpoint is incompatible "
    "with the current configuration."
)

MODEL.load_state_dict(
    BEST_CHECKPOINT["model"]
)

MODEL = MODEL.to(
    DEVICE
)

MODEL.eval()

# =========================================================================
# REQUIRED TOKENIZER / TEST SET
# =========================================================================
assert "tokenizer" in globals(), (
    "Tokenizer is unavailable. Run Cell 03 first."
)

assert "ACTIVE_TEST_RECORDS" in globals(), (
    "ACTIVE_TEST_RECORDS is unavailable. Run Cell 13 first."
)

assert len(
    ACTIVE_TEST_RECORDS
) > 0

assert callable(
    generate_architecture_A_continuation
), (
    "Architecture-A generation helper is unavailable. "
    "Run Cell 12 first."
)

# =========================================================================
# GENERATION ARTIFACTS
# =========================================================================
GENERATION_PROGRESS_PATH = (
    METRICS_DIR /
    "generation_progress_A.json"
)

PARTIAL_PREDICTION_PATH = (
    PREDICTIONS_DIR /
    "prediction_A_partial.json"
)

FINAL_PREDICTION_PATH = (
    PREDICTIONS_DIR /
    "prediction_A.json"
)

# =========================================================================
# TEST FIELD HELPERS
# =========================================================================
def _get_test_id(
    record
):
    return str(
        get_test_id(
            record
        )
    )


def _get_test_prompt(
    record
):
    return str(
        get_test_prompt(
            record
        )
    )


ACTIVE_TEST_IDS = [
    _get_test_id(
        record
    )
    for record in ACTIVE_TEST_RECORDS
]

assert len(
    ACTIVE_TEST_IDS
) == len(
    set(ACTIVE_TEST_IDS)
), (
    "Duplicate active test IDs detected."
)

# =========================================================================
# STABLE PER-EXAMPLE SEED
# =========================================================================
def _stable_generation_seed(
    test_id
):

    digest = hashlib.sha256(
        (
            f"{EXPERIMENT_ID}|"
            f"{ARCHITECTURE_NAME}|"
            f"{test_id}"
        ).encode("utf-8")
    ).digest()

    return (
        int.from_bytes(
            digest[:8],
            byteorder="big",
            signed=False,
        )
        % (2**31 - 1)
    )


# =========================================================================
# GENERATION CONFIGURATION
# =========================================================================
TARGET_WORDS = int(
    GENERATION_CONFIG[
        "target_words"
    ]
)

MAX_NEW_TOKENS = int(
    GENERATION_CONFIG[
        "max_new_tokens"
    ]
)

TEMPERATURE = float(
    GENERATION_CONFIG[
        "temperature"
    ]
)

TOP_P = float(
    GENERATION_CONFIG[
        "top_p"
    ]
)

TOP_K = int(
    GENERATION_CONFIG[
        "top_k"
    ]
)

REPETITION_PENALTY = float(
    GENERATION_CONFIG[
        "repetition_penalty"
    ]
)

# =========================================================================
# LOAD OR INITIALIZE GENERATION PROGRESS
# =========================================================================
predictions = {}

if (
    GENERATION_PROGRESS_PATH.is_file()
    and PARTIAL_PREDICTION_PATH.is_file()
):

    with open(
        GENERATION_PROGRESS_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        generation_progress = json.load(
            f
        )

    with open(
        PARTIAL_PREDICTION_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        predictions = json.load(
            f
        )

    # -------------------------------------------------------------
    # Compatibility checks.
    # -------------------------------------------------------------
    assert (
        generation_progress[
            "experiment_id"
        ]
        == EXPERIMENT_ID
    )

    assert (
        generation_progress[
            "architecture_name"
        ]
        == ARCHITECTURE_NAME
    )

    assert (
        generation_progress[
            "architecture_version"
        ]
        == ARCHITECTURE_VERSION
    )

    assert (
        generation_progress[
            "run_mode"
        ]
        == RUN_MODE
    )

    assert (
        int(
            generation_progress[
                "target_words"
            ]
        )
        == TARGET_WORDS
    )

    assert (
        int(
            generation_progress[
                "max_new_tokens"
            ]
        )
        == MAX_NEW_TOKENS
    )

    assert (
        int(
            generation_progress[
                "top_k"
            ]
        )
        == TOP_K
    )

    print(
        "♻️ Resuming existing generation."
    )

    print(
        f"   Already generated: "
        f"{len(predictions):,}/"
        f"{len(ACTIVE_TEST_RECORDS):,}"
    )

else:

    generation_progress = {
        "project": "CS781 Phase 1",

        "experiment_id": (
            EXPERIMENT_ID
        ),

        "architecture_name": (
            ARCHITECTURE_NAME
        ),

        "architecture_version": (
            ARCHITECTURE_VERSION
        ),

        "run_mode": RUN_MODE,

        "target_words": TARGET_WORDS,

        "max_new_tokens": MAX_NEW_TOKENS,

        "temperature": TEMPERATURE,

        "top_p": TOP_P,

        "top_k": TOP_K,

        "repetition_penalty": (
            REPETITION_PENALTY
        ),

        "completed_ids": [],

        "completed_count": 0,

        "status": "running",
    }

# =========================================================================
# RESUME VALIDATION
# =========================================================================
EXPECTED_ACTIVE_IDS = set(
    ACTIVE_TEST_IDS
)

# Never allow predictions for unknown IDs.
assert set(
    predictions.keys()
).issubset(
    EXPECTED_ACTIVE_IDS
), (
    "Generation checkpoint contains "
    "unknown test IDs."
)

# =========================================================================
# GENERATE REMAINING EXAMPLES
# =========================================================================
completed_ids = set(
    predictions.keys()
)

for index, record in enumerate(
    ACTIVE_TEST_RECORDS,
    start=1,
):

    test_id = _get_test_id(
        record
    )

    prompt = _get_test_prompt(
        record
    )

    # -------------------------------------------------------------
    # Skip already completed predictions.
    # -------------------------------------------------------------
    if test_id in completed_ids:

        continue

    # -------------------------------------------------------------
    # Deterministic example-local randomness.
    # -------------------------------------------------------------
    seed = _stable_generation_seed(
        test_id
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

    # -------------------------------------------------------------
    # Generate ONLY the continuation.
    # -------------------------------------------------------------
    continuation = (
        generate_architecture_A_continuation(
            prompt,
            target_words=TARGET_WORDS,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            repetition_penalty=(
                REPETITION_PENALTY
            ),
        )
    )

    continuation = str(
        continuation
    ).strip()

    # -------------------------------------------------------------
    # Validate output.
    # -------------------------------------------------------------
    assert continuation, (
        f"Empty continuation for test ID "
        f"{test_id}."
    )

    generated_word_count = (
        count_words(
            continuation
        )
    )

    assert (
        generated_word_count
        == TARGET_WORDS
    ), (
        f"Test ID {test_id}: expected "
        f"{TARGET_WORDS} words, got "
        f"{generated_word_count}."
    )

    # -------------------------------------------------------------
    # Store only the generated continuation.
    # -------------------------------------------------------------
    predictions[test_id] = {
        "text": continuation
    }

    completed_ids.add(
        test_id
    )

    # -------------------------------------------------------------
    # Persist immediately.
    # -------------------------------------------------------------
    generation_progress[
        "completed_ids"
    ] = sorted(
        completed_ids
    )

    generation_progress[
        "completed_count"
    ] = len(
        completed_ids
    )

    generation_progress[
        "last_completed_id"
    ] = test_id

    with open(
        PARTIAL_PREDICTION_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            predictions,
            f,
            ensure_ascii=False,
            indent=2,
        )

    with open(
        GENERATION_PROGRESS_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            generation_progress,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"[{index:>4}/"
        f"{len(ACTIVE_TEST_RECORDS):<4}] "
        f"ID={test_id} | "
        f"{generated_word_count} words"
    )

# =========================================================================
# FINAL VALIDATION
# =========================================================================
assert set(
    predictions.keys()
) == EXPECTED_ACTIVE_IDS, (
    "Prediction IDs do not exactly match "
    "the active official test IDs."
)

for test_id, prediction in (
    predictions.items()
):

    assert isinstance(
        prediction,
        dict,
    )

    assert set(
        prediction.keys()
    ) == {"text"}

    assert isinstance(
        prediction["text"],
        str,
    )

    assert count_words(
        prediction["text"]
    ) == TARGET_WORDS

# =========================================================================
# WRITE FINAL PREDICTION JSON
# =========================================================================
with open(
    FINAL_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

generation_progress[
    "completed_ids"
] = sorted(
    completed_ids
)

generation_progress[
    "completed_count"
] = len(
    completed_ids
)

generation_progress[
    "status"
] = "complete"

with open(
    GENERATION_PROGRESS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        generation_progress,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =========================================================================
# SUMMARY
# =========================================================================
print()
print(
    "✅ Architecture-A generation complete."
)

print(
    f"Run mode            : "
    f"{RUN_MODE}"
)

print(
    f"Predictions         : "
    f"{len(predictions):,}"
)

print(
    f"Words / prediction  : "
    f"{TARGET_WORDS}"
)

print(
    f"Maximum new tokens  : "
    f"{MAX_NEW_TOKENS}"
)

print(
    f"Temperature         : "
    f"{TEMPERATURE}"
)

print(
    f"Top-p               : "
    f"{TOP_P}"
)

print(
    f"Top-k               : "
    f"{TOP_K}"
)

print(
    f"Prediction file     : "
    f"{FINAL_PREDICTION_PATH}"
)

print(
    f"Progress file       : "
    f"{GENERATION_PROGRESS_PATH}"
)

🚀 Starting Architecture-A official-test generation...
♻️ Resuming existing generation.
   Already generated: 1,000/1,000

✅ Architecture-A generation complete.
Run mode            : production
Predictions         : 1,000
Words / prediction  : 128
Maximum new tokens  : 1536
Temperature         : 0.1
Top-p               : 0.95
Top-k               : 50
Prediction file     : /kaggle/working/architecture_A/predictions/prediction_A.json
Progress file       : /kaggle/working/architecture_A/metrics/generation_progress_A.json


In [22]:
# =========================================================================
# Cell 15 — Validate and package Architecture-A submission
# DECISION: REPLACE
#
# IMPORTANT:
#   Codabench expects:
#
#   {
#       "example_id_1": {
#           "text": "128-word continuation"
#       },
#       "example_id_2": {
#           "text": "128-word continuation"
#       }
#   }
#
#   NOT:
#
#   [
#       {"id": "example_id_1", "text": "..."},
#       ...
#   ]
# =========================================================================

import json
import zipfile
import shutil
from pathlib import Path


print(
    "📦 Preparing Architecture-A submission package..."
)


# =========================================================================
# PATHS
# =========================================================================

ARCHITECTURE_A_ROOT = Path(
    "/kaggle/working/architecture_A"
)

PREDICTIONS_DIR = (
    ARCHITECTURE_A_ROOT / "predictions"
)

METRICS_DIR = (
    ARCHITECTURE_A_ROOT / "metrics"
)

SUBMISSION_DIR = (
    ARCHITECTURE_A_ROOT / "submission_package"
)

PREDICTION_FINAL_PATH = (
    PREDICTIONS_DIR / "prediction_A.json"
)

SUBMISSION_PREDICTION_PATH = (
    SUBMISSION_DIR / "prediction.json"
)

SUBMISSION_ZIP_PATH = (
    ARCHITECTURE_A_ROOT / "submission.zip"
)

SUBMISSION_MANIFEST_PATH = (
    METRICS_DIR / "submission_manifest_A.json"
)


# =========================================================================
# CHECK FINAL PREDICTION FILE
# =========================================================================

assert PREDICTION_FINAL_PATH.is_file(), (
    f"❌ Final Architecture-A prediction file not found:\n"
    f"{PREDICTION_FINAL_PATH}\n\n"
    "Run Architecture-A Cell 14 successfully first."
)


# =========================================================================
# LOAD PREDICTIONS
# =========================================================================

with open(
    PREDICTION_FINAL_PATH,
    "r",
    encoding="utf-8",
) as f:

    raw_prediction_data = json.load(f)


print()
print(
    f"Raw prediction type : "
    f"{type(raw_prediction_data).__name__}"
)


# =========================================================================
# NORMALIZE INTO REQUIRED DICTIONARY FORMAT
# =========================================================================
#
# Final required format:
#
# {
#     "1": {
#         "text": "..."
#     },
#     "2": {
#         "text": "..."
#     }
# }
#
# We support either:
#
#   1. Existing dictionary:
#      {"1": {"text": "..."}}
#
#   2. List of records:
#      [{"id": "1", "text": "..."}]
#
#   3. Wrapper structures.
# =========================================================================

def normalize_to_prediction_dict(
    raw,
):

    # ---------------------------------------------------------------------
    # Case 1: dictionary
    # ---------------------------------------------------------------------
    if isinstance(
        raw,
        dict,
    ):

        # Already in required format:
        # {"1": {"text": "..."}}
        direct_mapping = True

        for key, value in raw.items():

            if not isinstance(
                value,
                dict,
            ):
                direct_mapping = False
                break

            text_value = None

            for text_key in (
                "text",
                "continuation",
                "generated_text",
            ):

                if (
                    text_key in value
                    and isinstance(
                        value[text_key],
                        str,
                    )
                ):

                    text_value = (
                        value[text_key]
                    )

                    break

            if text_value is None:

                direct_mapping = False
                break

        if direct_mapping:

            return {
                str(key): {
                    "text": (
                        value["text"]
                        if "text" in value
                        else (
                            value.get(
                                "continuation",
                                value.get(
                                    "generated_text"
                                ),
                            )
                        )
                    )
                }
                for key, value in raw.items()
            }

        # -----------------------------------------------------------------
        # Wrapper dictionary
        # -----------------------------------------------------------------
        for wrapper_key in (
            "predictions",
            "results",
            "records",
            "data",
            "outputs",
        ):

            if isinstance(
                raw.get(wrapper_key),
                list,
            ):

                return normalize_to_prediction_dict(
                    raw[wrapper_key]
                )

        # -----------------------------------------------------------------
        # Single record
        # -----------------------------------------------------------------
        if (
            "id" in raw
            and (
                "text" in raw
                or "continuation" in raw
                or "generated_text" in raw
            )
        ):

            return normalize_to_prediction_dict(
                [raw]
            )

        # -----------------------------------------------------------------
        # Dictionary mapping ID -> string
        # -----------------------------------------------------------------
        string_mapping = True

        converted = {}

        for record_id, value in raw.items():

            if not isinstance(
                value,
                str,
            ):

                string_mapping = False
                break

            converted[
                str(record_id)
            ] = {
                "text": value
            }

        if string_mapping and converted:

            return converted


        # -----------------------------------------------------------------
        # Dictionary mapping ID -> record
        # -----------------------------------------------------------------
        converted = {}

        valid_record_mapping = True

        for record_id, value in raw.items():

            if not isinstance(
                value,
                dict,
            ):

                valid_record_mapping = False
                break

            text_value = None

            for text_key in (
                "text",
                "continuation",
                "generated_text",
            ):

                if (
                    text_key in value
                    and isinstance(
                        value[text_key],
                        str,
                    )
                ):

                    text_value = (
                        value[text_key]
                    )

                    break

            if text_value is None:

                valid_record_mapping = False
                break

            converted[
                str(record_id)
            ] = {
                "text": text_value
            }

        if (
            valid_record_mapping
            and converted
        ):

            return converted


    # ---------------------------------------------------------------------
    # Case 2: list of records
    # ---------------------------------------------------------------------
    if isinstance(
        raw,
        list,
    ):

        converted = {}

        for index, record in enumerate(
            raw
        ):

            assert isinstance(
                record,
                dict,
            ), (
                f"Prediction {index} is not a dictionary."
            )

            assert "id" in record, (
                f"Prediction {index} is missing 'id'."
            )

            prediction_id = str(
                record["id"]
            )

            text_value = None

            for text_key in (
                "text",
                "continuation",
                "generated_text",
            ):

                if (
                    text_key in record
                    and isinstance(
                        record[text_key],
                        str,
                    )
                ):

                    text_value = (
                        record[text_key]
                    )

                    break

            assert text_value is not None, (
                f"Prediction {prediction_id} "
                "has no text field."
            )

            assert prediction_id not in converted, (
                f"Duplicate prediction ID: "
                f"{prediction_id}"
            )

            converted[
                prediction_id
            ] = {
                "text": text_value
            }

        return converted


    raise ValueError(
        "Could not normalize Architecture-A predictions."
    )


prediction_by_id = (
    normalize_to_prediction_dict(
        raw_prediction_data
    )
)


assert isinstance(
    prediction_by_id,
    dict,
)

print(
    f"Normalized predictions : "
    f"{len(prediction_by_id):,}"
)


# =========================================================================
# EXPECTED COUNT
# =========================================================================

if RUN_MODE == "fast":

    expected_count = len(
        ACTIVE_TEST_RECORDS
    )

else:

    expected_count = 1000


assert len(
    prediction_by_id
) == expected_count, (
    f"Expected {expected_count} predictions "
    f"in {RUN_MODE} mode, but found "
    f"{len(prediction_by_id)}."
)


# =========================================================================
# EXPECTED OFFICIAL TEST IDS
# =========================================================================

expected_ids = [
    str(
        get_test_id(record)
    )
    for record in ACTIVE_TEST_RECORDS
]

expected_id_set = set(
    expected_ids
)

actual_id_set = set(
    prediction_by_id.keys()
)

assert actual_id_set == expected_id_set, (
    "Prediction IDs do not exactly match "
    "the active official-test IDs.\n"
    f"Missing: "
    f"{sorted(expected_id_set - actual_id_set)[:10]}\n"
    f"Unexpected: "
    f"{sorted(actual_id_set - expected_id_set)[:10]}"
)


# =========================================================================
# VALIDATE EVERY PREDICTION
# =========================================================================

target_words = int(
    GENERATION_CONFIG[
        "target_words"
    ]
)

for prediction_id in expected_ids:

    assert prediction_id in prediction_by_id

    prediction = (
        prediction_by_id[
            prediction_id
        ]
    )

    assert isinstance(
        prediction,
        dict,
    )

    assert set(
        prediction.keys()
    ) == {
        "text"
    }, (
        f"Prediction {prediction_id} "
        "must contain only 'text'."
    )

    prediction_text = (
        prediction["text"]
    )

    assert isinstance(
        prediction_text,
        str,
    ), (
        f"Prediction {prediction_id} "
        "text must be a string."
    )

    word_count = len(
        prediction_text.strip().split()
    )

    assert word_count == target_words, (
        f"Prediction {prediction_id}: "
        f"expected {target_words} words, "
        f"found {word_count}."
    )


# =========================================================================
# PRESERVE OFFICIAL ORDER
# =========================================================================

canonical_prediction_data = {
    prediction_id: prediction_by_id[
        prediction_id
    ]
    for prediction_id in expected_ids
}


assert len(
    canonical_prediction_data
) == expected_count


# =========================================================================
# CREATE CLEAN SUBMISSION DIRECTORY
# =========================================================================

ARCHITECTURE_A_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if SUBMISSION_DIR.exists():

    shutil.rmtree(
        SUBMISSION_DIR
    )

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================================
# WRITE OFFICIAL prediction.json
# =========================================================================

with open(
    SUBMISSION_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        canonical_prediction_data,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# IMMEDIATE FORMAT VERIFICATION
# =========================================================================

with open(
    SUBMISSION_PREDICTION_PATH,
    "r",
    encoding="utf-8",
) as f:

    verify_prediction = json.load(
        f
    )


assert isinstance(
    verify_prediction,
    dict,
), (
    "❌ submission prediction.json is not a JSON object."
)

assert len(
    verify_prediction
) == expected_count


for prediction_id in expected_ids:

    assert prediction_id in verify_prediction

    assert (
        isinstance(
            verify_prediction[
                prediction_id
            ],
            dict,
        )
    )

    assert set(
        verify_prediction[
            prediction_id
        ].keys()
    ) == {
        "text"
    }


print()
print(
    "✅ prediction.json format verified:"
)

print(
    "   Top-level type : JSON object"
)

print(
    f"   Predictions    : {len(verify_prediction):,}"
)

print(
    f"   Words/prediction: {target_words}"
)


# =========================================================================
# CREATE submission.zip
# =========================================================================

if SUBMISSION_ZIP_PATH.exists():

    SUBMISSION_ZIP_PATH.unlink()


with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    # prediction.json MUST be at ZIP root.
    zf.write(
        SUBMISSION_PREDICTION_PATH,
        arcname="prediction.json",
    )


# =========================================================================
# VERIFY ZIP
# =========================================================================

with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH,
    mode="r",
) as zf:

    zip_names = zf.namelist()

    assert zip_names == [
        "prediction.json"
    ], (
        f"Incorrect ZIP structure: "
        f"{zip_names}"
    )

    with zf.open(
        "prediction.json"
    ) as f:

        zipped_prediction_data = (
            json.loads(
                f.read().decode(
                    "utf-8"
                )
            )
        )


assert isinstance(
    zipped_prediction_data,
    dict,
)

assert (
    zipped_prediction_data
    == canonical_prediction_data
), (
    "ZIP prediction.json does not match "
    "the canonical prediction data."
)


# =========================================================================
# SAVE MANIFEST
# =========================================================================

submission_manifest = {

    "project":
        "CS781 Phase 1",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "prediction_count":
        len(canonical_prediction_data),

    "target_words":
        target_words,

    "prediction_file":
        str(PREDICTION_FINAL_PATH),

    "submission_prediction_file":
        str(SUBMISSION_PREDICTION_PATH),

    "submission_zip":
        str(SUBMISSION_ZIP_PATH),

    "zip_contents":
        zip_names,

    "validation": {

        "top_level_json_object":
            True,

        "unique_ids":
            True,

        "ids_match_active_test_set":
            True,

        "official_test_order_preserved":
            True,

        "exact_word_count":
            True,

        "zip_structure_verified":
            True,

        "zip_content_verified":
            True,
    },
}


with open(
    SUBMISSION_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        submission_manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# FINAL SUMMARY
# =========================================================================

zip_size_mb = (
    SUBMISSION_ZIP_PATH.stat().st_size
    / (1024 ** 2)
)


print()
print(
    "✅ Architecture-A submission packaged successfully."
)

print()
print(
    f"Run mode            : {RUN_MODE}"
)

print(
    f"Predictions         : "
    f"{len(canonical_prediction_data):,}"
)

print(
    f"Words / prediction  : "
    f"{target_words}"
)

print(
    "Top-level JSON      : OBJECT"
)

print(
    "ZIP contents        : prediction.json"
)

print(
    f"Prediction JSON     : "
    f"{SUBMISSION_PREDICTION_PATH}"
)

print(
    f"Submission ZIP      : "
    f"{SUBMISSION_ZIP_PATH}"
)

print(
    f"ZIP size            : "
    f"{zip_size_mb:.3f} MB"
)

print(
    f"Manifest            : "
    f"{SUBMISSION_MANIFEST_PATH}"
)

📦 Preparing Architecture-A submission package...

Raw prediction type : dict
Normalized predictions : 1,000

✅ prediction.json format verified:
   Top-level type : JSON object
   Predictions    : 1,000
   Words/prediction: 128

✅ Architecture-A submission packaged successfully.

Run mode            : production
Predictions         : 1,000
Words / prediction  : 128
Top-level JSON      : OBJECT
ZIP contents        : prediction.json
Prediction JSON     : /kaggle/working/architecture_A/submission_package/prediction.json
Submission ZIP      : /kaggle/working/architecture_A/submission.zip
ZIP size            : 0.051 MB
Manifest            : /kaggle/working/architecture_A/metrics/submission_manifest_A.json


In [23]:
# Cell 16 — Final Architecture-A artifact inventory and verification
# DECISION: UPDATE

import json
import zipfile
from pathlib import Path


print("🔍 Finalizing Architecture-A artifact inventory...")


# =========================================================================
# ROOT DIRECTORIES
# =========================================================================

ARCHITECTURE_A_ROOT = Path(
    "/kaggle/working/architecture_A"
)

CONFIGS_DIR = ARCHITECTURE_A_ROOT / "configs"
CHECKPOINTS_DIR = ARCHITECTURE_A_ROOT / "checkpoints"
METRICS_DIR = ARCHITECTURE_A_ROOT / "metrics"
PREDICTIONS_DIR = ARCHITECTURE_A_ROOT / "predictions"


assert ARCHITECTURE_A_ROOT.exists(), (
    f"Architecture-A root not found:\n"
    f"{ARCHITECTURE_A_ROOT}"
)


# =========================================================================
# CORE ARTIFACTS
# =========================================================================

core_artifacts = {

    "best_checkpoint":
        CHECKPOINTS_DIR / "best.pt",

    "latest_checkpoint":
        CHECKPOINTS_DIR / "latest.pt",

    "checkpoint_index":
        METRICS_DIR / "checkpoint_index_A.json",

    "selected_checkpoint":
        METRICS_DIR / "selected_checkpoint_A.json",

    "model_metadata":
        ARCHITECTURE_A_ROOT / "model_A_metadata.json",

    "final_model":
        ARCHITECTURE_A_ROOT / "model_A_final.pt",

    "state_dict":
        ARCHITECTURE_A_ROOT / "model_A_state_dict.pt",

    "generation_config":
        CONFIGS_DIR / "architecture_A_generation_config.json",

    "test_manifest":
        CONFIGS_DIR / "official_test_manifest_A.json",

    "generation_progress":
        METRICS_DIR / "generation_progress_A.json",

    "prediction":
        PREDICTIONS_DIR / "prediction_A.json",

    "submission_zip":
        ARCHITECTURE_A_ROOT / "submission.zip",

    "submission_manifest":
        METRICS_DIR / "submission_manifest_A.json",
}


# =========================================================================
# OPTIONAL ARTIFACTS
# =========================================================================

optional_artifacts = {

    "config_fingerprint_json":
        CONFIGS_DIR / "architecture_A_config_fingerprint.json",
}


# =========================================================================
# VERIFY REQUIRED ARTIFACTS
# =========================================================================

missing_core = [
    (name, path)
    for name, path in core_artifacts.items()
    if not path.exists()
]

assert not missing_core, (
    "Missing required Architecture-A artifacts:\n"
    + "\n".join(
        f"- {name}: {path}"
        for name, path in missing_core
    )
)


# =========================================================================
# LOAD MODEL METADATA
# =========================================================================

with open(
    core_artifacts["model_metadata"],
    "r",
    encoding="utf-8",
) as f:

    model_metadata = json.load(f)


assert isinstance(
    model_metadata,
    dict,
), "model_A_metadata.json must contain a JSON object."


# =========================================================================
# DISPLAY ACTUAL METADATA SCHEMA
# =========================================================================

print()
print("Model metadata keys:")

for key in model_metadata.keys():
    print(f"  • {key}")


# =========================================================================
# LOAD CHECKPOINT INDEX
# =========================================================================

with open(
    core_artifacts["checkpoint_index"],
    "r",
    encoding="utf-8",
) as f:

    checkpoint_index = json.load(f)


assert isinstance(
    checkpoint_index,
    dict,
), "checkpoint_index_A.json must contain a JSON object."


# =========================================================================
# LOAD SUBMISSION MANIFEST
# =========================================================================

with open(
    core_artifacts["submission_manifest"],
    "r",
    encoding="utf-8",
) as f:

    submission_manifest = json.load(f)


assert isinstance(
    submission_manifest,
    dict,
), "submission_manifest_A.json must contain a JSON object."


# =========================================================================
# VERIFY SUBMISSION ZIP
# =========================================================================

with zipfile.ZipFile(
    core_artifacts["submission_zip"],
    mode="r",
) as zf:

    zip_contents = zf.namelist()

    assert zip_contents == [
        "prediction.json"
    ], (
        "submission.zip must contain exactly "
        "'prediction.json' at ZIP root.\n"
        f"Found: {zip_contents}"
    )


# =========================================================================
# VERIFY SUBMISSION COUNT
# =========================================================================

submission_prediction_count = (
    submission_manifest.get(
        "prediction_count"
    )
)

assert submission_prediction_count is not None, (
    "submission_manifest_A.json does not contain "
    "'prediction_count'."
)

if RUN_MODE == "fast":

    expected_prediction_count = len(
        ACTIVE_TEST_RECORDS
    )

else:

    expected_prediction_count = 1000


assert (
    submission_prediction_count
    == expected_prediction_count
), (
    f"Submission prediction count mismatch: "
    f"{submission_prediction_count} != "
    f"{expected_prediction_count}"
)


# =========================================================================
# VERIFY IDENTITY USING ACTUAL METADATA SCHEMA
# =========================================================================

assert (
    model_metadata.get("experiment_id")
    == EXPERIMENT_ID
), "Model metadata experiment_id mismatch."

assert (
    model_metadata.get("architecture_name")
    == ARCHITECTURE_NAME
), "Model metadata architecture_name mismatch."

assert (
    model_metadata.get("architecture_version")
    == ARCHITECTURE_VERSION
), "Model metadata architecture_version mismatch."

assert (
    model_metadata.get("run_mode")
    == RUN_MODE
), "Model metadata run_mode mismatch."


# =========================================================================
# EXTRACT STORED METRICS USING ACTUAL CELL-10 FIELD NAMES
# =========================================================================

parameter_count = model_metadata.get(
    "parameter_count"
)

parameter_millions = model_metadata.get(
    "parameter_millions"
)

best_step = model_metadata.get(
    "best_validation_step"
)

best_validation_loss = model_metadata.get(
    "best_validation_loss"
)

best_validation_perplexity = model_metadata.get(
    "best_validation_perplexity"
)

tokens_seen = model_metadata.get(
    "tokens_seen_at_best"
)

checkpoint_count = model_metadata.get(
    "best_checkpoint_count"
)

early_stopping_info = model_metadata.get(
    "early_stopping"
)

config_fingerprint = model_metadata.get(
    "config_fingerprint"
)


# =========================================================================
# VALIDATE IMPORTANT STORED VALUES
# =========================================================================

assert parameter_count is not None, (
    "parameter_count missing from model metadata."
)

assert best_step is not None, (
    "best_validation_step missing from model metadata."
)

assert best_validation_loss is not None, (
    "best_validation_loss missing from model metadata."
)

assert best_validation_perplexity is not None, (
    "best_validation_perplexity missing from model metadata."
)

assert tokens_seen is not None, (
    "tokens_seen_at_best missing from model metadata."
)

assert checkpoint_count is not None, (
    "best_checkpoint_count missing from model metadata."
)


# =========================================================================
# CHECKPOINT INDEX
# =========================================================================

best_checkpoint_entries = (
    checkpoint_index.get(
        "best_checkpoints",
        []
    )
)

if not isinstance(
    best_checkpoint_entries,
    list,
):

    best_checkpoint_entries = []


# =========================================================================
# GENERATION CONFIG
# =========================================================================

with open(
    core_artifacts["generation_config"],
    "r",
    encoding="utf-8",
) as f:

    generation_manifest = json.load(f)


assert (
    generation_manifest.get("run_mode")
    == RUN_MODE
), "Generation configuration run-mode mismatch."


# =========================================================================
# TEST MANIFEST
# =========================================================================

with open(
    core_artifacts["test_manifest"],
    "r",
    encoding="utf-8",
) as f:

    test_manifest = json.load(f)


assert isinstance(
    test_manifest,
    dict,
), "Official test manifest must contain a JSON object."


# =========================================================================
# INVENTORY
# =========================================================================

all_artifacts = {}

for name, path in core_artifacts.items():

    all_artifacts[name] = {
        "required": True,
        "exists": True,
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "size_mb": round(
            path.stat().st_size / (1024 ** 2),
            4,
        ),
    }


for name, path in optional_artifacts.items():

    exists = path.exists()

    all_artifacts[name] = {
        "required": False,
        "exists": exists,
        "path": str(path),
        "size_bytes": (
            path.stat().st_size
            if exists
            else 0
        ),
        "size_mb": (
            round(
                path.stat().st_size / (1024 ** 2),
                4,
            )
            if exists
            else 0.0
        ),
    }


# =========================================================================
# SAVE FINAL INVENTORY
# =========================================================================

inventory_payload = {

    "project": "CS781 Phase 1",

    "experiment_id": EXPERIMENT_ID,

    "architecture": ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "model_parameters":
        parameter_count,

    "model_parameters_millions":
        parameter_millions,

    "best_validation_step":
        best_step,

    "best_validation_loss":
        best_validation_loss,

    "best_validation_perplexity":
        best_validation_perplexity,

    "tokens_seen_at_best":
        tokens_seen,

    "best_checkpoint_count":
        checkpoint_count,

    "checkpoint_index_best_entries":
        len(best_checkpoint_entries),

    "early_stopping":
        early_stopping_info,

    "config_fingerprint_embedded":
        config_fingerprint is not None,

    "prediction_count":
        submission_prediction_count,

    "zip_contents":
        zip_contents,

    "required_artifact_count":
        len(core_artifacts),

    "optional_artifact_count":
        len(optional_artifacts),

    "optional_artifacts_missing": [
        name
        for name, path in optional_artifacts.items()
        if not path.exists()
    ],

    "artifacts":
        all_artifacts,
}


INVENTORY_PATH = (
    METRICS_DIR
    / "architecture_A_artifact_inventory.json"
)

with open(
    INVENTORY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        inventory_payload,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# FINAL SUMMARY
# =========================================================================

print()
print("✅ Architecture-A inventory verified.")
print()

print(
    f"Run mode              : {RUN_MODE}"
)

print(
    f"Architecture           : "
    f"{ARCHITECTURE_NAME}"
)

print(
    f"Version                : "
    f"{ARCHITECTURE_VERSION}"
)

print(
    f"Required artifacts     : "
    f"{len(core_artifacts)}/{len(core_artifacts)}"
)

print(
    f"Model parameters       : "
    f"{parameter_count:,} "
    f"({parameter_millions}M)"
)

print(
    f"Best validation step  : "
    f"{best_step}"
)

print(
    f"Best validation loss  : "
    f"{best_validation_loss}"
)

print(
    f"Best validation PPL   : "
    f"{best_validation_perplexity}"
)

print(
    f"Tokens seen at best   : "
    f"{tokens_seen:,}"
)

print(
    f"Best checkpoint count : "
    f"{checkpoint_count}"
)

print(
    f"Checkpoint index      : "
    f"{len(best_checkpoint_entries)} entries"
)

print(
    f"Submission predictions: "
    f"{submission_prediction_count}"
)

print(
    f"Config fingerprint    : "
    f"{'embedded' if config_fingerprint is not None else 'missing'}"
)

print(
    f"Submission ZIP        : "
    f"{core_artifacts['submission_zip']}"
)

print(
    f"Inventory             : "
    f"{INVENTORY_PATH}"
)


missing_optional = [
    name
    for name, path in optional_artifacts.items()
    if not path.exists()
]

if missing_optional:

    print()
    print("ℹ️ Optional artifacts not present:")

    for name in missing_optional:

        print(
            f"  - {name}"
        )


print()
print("📁 Core Architecture-A artifacts:")

for name, data in all_artifacts.items():

    if data["required"]:

        print(
            f"  ✅ {name:24s} "
            f"{data['size_mb']:.4f} MB"
        )

🔍 Finalizing Architecture-A artifact inventory...

Model metadata keys:
  • project
  • experiment_id
  • architecture_name
  • architecture_version
  • run_mode
  • model_type
  • parameter_count
  • parameter_millions
  • model_config
  • best_checkpoint
  • best_validation_step
  • best_validation_loss
  • best_validation_perplexity
  • tokens_seen_at_best
  • best_checkpoint_count
  • early_stopping
  • source_probabilities
  • tokenizer_vocab_size
  • config_fingerprint

✅ Architecture-A inventory verified.

Run mode              : production
Architecture           : Architecture-A-Decoder-Only-Transformer
Version                : A-v2-early-stop-dual-mode
Required artifacts     : 13/13
Model parameters       : 125,028,864 (125.028864M)
Best validation step  : 4500
Best validation loss  : 3.0104399347305297
Best validation PPL   : 20.296327020585853
Tokens seen at best   : 73,728,000
Best checkpoint count : 8
Checkpoint index      : 0 entries
Submission predictions: 1000
Config fi